## Setup: Install and Import Core Libraries

This cell installs the required Python packages and imports `pandas`, the primary data manipulation library used throughout this analysis. `matplotlib` is also installed here for later visualisation use.

In [3]:
%pip install pandas
import pandas as pd
%pip install matplotlib


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


A specific version of NumPy (1.24.3) is pinned here to ensure compatibility with downstream packages such as SHAP, which can break with newer NumPy releases.

In [43]:
%pip install numpy==1.24.3

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
daal4py 2021.5.0 requires daal==2021.4.0, which is not installed.
scipy 1.7.3 requires numpy<1.23.0,>=1.16.5, but you have numpy 1.24.3 which is incompatible.
numba 0.55.1 requires numpy<1.22,>=1.18, but you have numpy 1.24.3 which is incompatible.


  Using cached numpy-1.24.3-cp39-cp39-win_amd64.whl (14.9 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.21.6
    Uninstalling numpy-1.21.6:
      Successfully uninstalled numpy-1.21.6


Imports `matplotlib.pyplot` for plotting. Kept separate from the install cell above since the import only needs to happen once per kernel session.

In [5]:
import matplotlib.pyplot as plt

## 1. Load Data

The dataset is the OSBAP (Option-Adjusted Spread Bond Analytics Panel), stored as a pickled DataFrame. This cell loads it and prints basic structural information — shape, column names, a row preview, data types, and summary statistics — to confirm the data loaded correctly and to understand its layout.

In [6]:
df=pd.read_pickle(r"C:\Users\vishn\OneDrive\Desktop\RSM MScBA\Thesis\OSBAP_ML_Panel_Oct_2024.pkl")
print("Shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nFirst few rows:")
print(df.head())
print("\nData types:")
print(df.dtypes)
print("\nBasic summary stats:")
print(df.describe())

Shape: (1102569, 381)

Column names:
['date', 'ID', 'cusip', 'issuer_cusip', 'gvkey', 'permno', 'permco', 'dnr_sample', 'fx', 'country', 'ice_idx', 'RATING_NUM', 'VW', '1_age', '2_coupon', '3_faceval', '6_duration', '12_mom6', '13_mom6ind', '14_mom6xrtg', '18_spread', '22_rating', '24_skew', '25_mom6mspread', '27_volatility', '28_VaR', '29_vixbeta', '_mom3mspread', '_ratingxspread', '_ave12mspread', '_spreadvol', '_value', '_impliedspread', 'strevb', 'tmt', 'sizeb', 'dspread', 'dts', 'kurt', 'swap_spread', 'yield', 'yld_to_worst', 'convexity', 'ltrev48_12', 'ltrev30_6', 'bond_price', 'emp_value', 'struc_value', 'bond_mom_ipr', 'stock_mom_ipr', 'mkt_lev_ipr', 'gross_prof_ipr', '26_spr_to_d2d', 'niq_su', 'ret_6_1', 'ret_12_1', 'saleq_su', 'tax_gr1a', 'ni_inc8q', 'prc_highprc_252d', 'resff3_6_1', 'resff3_12_1', 'be_me', 'debt_me', 'at_me', 'ret_60_12', 'ni_me', 'fcf_me', 'div12m_me', 'eqpo_me', 'eqnpo_me', 'sale_gr3', 'sale_gr1', 'ebitda_mev', 'sale_me', 'ocf_me', 'ival_me', 'bev_mev', 'n

## 2. Exploratory Data Analysis — Coverage and Key Variables

This cell examines the key dimensions of the panel: the time period covered, the number of unique bonds (identified by CUSIP), and the distribution and completeness of the target variable (`18_spread`, the option-adjusted spread in basis points). It also checks for missing values in the most important columns, the distribution of credit ratings, and the split between investment-grade (IG, `ice_idx = 'c0a0'`) and high-yield (HY, `ice_idx = 'H0A0'`) bonds.

In [7]:
import pandas as pd

# Check date range
print("Date range:", df['date'].min(), "to", df['date'].max())

# Check number of unique bonds
print("Unique bonds:", df['cusip'].nunique())

# Check the spread variable
print("\nSpread summary:")
print(df['18_spread'].describe())

# Check missing values in key columns
key_cols = ['18_spread', '6_duration', 'RATING_NUM', '22_rating', 
            '27_volatility', 'date', 'cusip']
print("\nMissing values in key columns:")
print(df[key_cols].isnull().sum())

# Rating distribution
print("\nRating distribution:")
print(df['RATING_NUM'].value_counts().sort_index())

# Check IG vs HY split using rating
print("\nIG vs HY count:")
print(df['ice_idx'].value_counts())


Date range: 2002-07-31 00:00:00 to 2022-11-30 00:00:00
Unique bonds: 19768

Spread summary:
count    1.102569e+06
mean     4.284872e-03
std      5.750112e-01
min     -1.000000e+00
25%     -4.923214e-01
50%      4.671533e-03
75%      5.019655e-01
max      1.000000e+00
Name: 18_spread, dtype: float64

Missing values in key columns:
18_spread        0
6_duration       0
RATING_NUM       0
22_rating        0
27_volatility    0
date             0
cusip            0
dtype: int64

Rating distribution:
RATING_NUM
1      12892
2       5298
3      15389
4      41518
5      81537
6     153937
7     122326
8     150719
9     180921
10    111913
11     40162
12     36271
13     36341
14     30006
15     29288
16     23834
17     15496
18      7349
19      3269
20      3175
21       851
22        77
Name: count, dtype: int64

IG vs HY count:
ice_idx
c0a0    876534
H0A0    226035
Name: count, dtype: int64


## 3. Exploratory Data Analysis — Spread Variables

This cell inventories all spread-related columns in the dataset and examines the raw yield and `dspread` (spread duration) variables to understand what spread measures are available. A sample time series for a single bond is printed to verify the panel structure.

In [8]:
# Check if there are other spread-related columns
spread_cols = [col for col in df.columns if 'spread' in col.lower() 
               or 'spr' in col.lower()]
print("Spread-related columns:")
print(spread_cols)

# Check the yield column which might be raw
print("\nYield summary:")
print(df['yield'].describe())

# Check dspread
print("\ndspread summary:")
print(df['dspread'].describe())

# Check the TRACE stage 1 columns
# The raw spread might be under a different name
print("\nFirst 5 rows of spread-related columns:")
print(df[spread_cols].head())

# Also check what ice_idx values mean
print("\nSample of key columns for one bond:")
sample = df[df['cusip'] == '001031AC7'][['date', '18_spread', 
             'yield', 'dspread', 'RATING_NUM', 'ice_idx']].head(10)
print(sample)

Spread-related columns:
['18_spread', '25_mom6mspread', '_mom3mspread', '_ratingxspread', '_ave12mspread', '_spreadvol', '_impliedspread', 'dspread', 'swap_spread', '26_spr_to_d2d', 'mispricing_perf', 'mispricing_mgmt', 'CPVolSpread', 'RIVolSpread', 'dCPVolSpread']

Yield summary:
count    1.102569e+06
mean     5.344017e-03
std      5.741546e-01
min     -1.000000e+00
25%     -4.909299e-01
50%      6.031146e-03
75%      5.022604e-01
max      1.000000e+00
Name: yield, dtype: float64

dspread summary:
count    1.102569e+06
mean     2.058839e-03
std      5.751295e-01
min     -1.000000e+00
25%     -4.938354e-01
50%      3.615329e-03
75%      4.987987e-01
max      1.000000e+00
Name: dspread, dtype: float64

First 5 rows of spread-related columns:
   18_spread  25_mom6mspread  _mom3mspread  _ratingxspread  _ave12mspread  \
0   0.807932    0.000000e+00      0.000000        0.828806   0.000000e+00   
1   0.872933    0.000000e+00      0.000000        0.880393   2.486325e-04   
2   0.918195    0.

In [9]:
%pip install pandas_datareader

Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: python-dateutil>=2.8.2 in c:\users\vishn\anaconda3\lib\site-packages (from pandas>=0.23->pandas_datareader) (2.8.2)



In [14]:
%pip install fredapi

Note: you may need to restart the kernel to use updated packages.


In [16]:
import fredapi
from fredapi import Fred
import pandas as pd
import numpy as np

FRED_API_KEY = 'your_api_key_here'  # Replace with your key
fred = Fred(api_key="55f1b480f1405d9d3fa580780bd9bc5e")

start = '2002-01-01'
end   = '2022-12-31'

# The four most important macro-financial variables from CGM (2001)
series_dict = {
    'vix':      'VIXCLS',   # Aggregate volatility — risk appetite
    'tsy10':    'DGS10',    # Treasury yield level — interest rate channel
    'term_spr': 'T10Y2Y',   # Yield curve slope — term structure channel
    'ted':      'TEDRATE',  # TED spread — funding liquidity channel (discontinued 2023, but available through 2022)
}

macro_raw = {}
for name, ticker in series_dict.items():
    try:
        s = fred.get_series(ticker, observation_start=start, observation_end=end)
        s.name = name
        macro_raw[name] = s
        print(f"Downloaded {name} ({ticker}): {len(s)} obs")
    except Exception as e:
        print(f"Failed {name}: {e}")

macro_df = pd.concat(macro_raw.values(), axis=1)

# Resample to month-end to match bond panel
macro_df = macro_df.resample('M').last()

# Construct transformations following CGM (2001)
macro_df['tsy10_ch'] = macro_df['tsy10'].diff(1)   # Change in yield
macro_df['vix_ch']   = macro_df['vix'].diff(1)     # Change in VIX
macro_df['term_spr'] = macro_df['term_spr']         # Level of slope
macro_df['ted_lvl']  = macro_df['ted']              # TED spread level

# Keep only the transformed versions
macro_final = macro_df[['vix', 'vix_ch', 'tsy10', 'tsy10_ch',
                         'term_spr', 'ted_lvl']].copy()

print("\nFinal macro shape:", macro_final.shape)
print("\nMissing values:")
print(macro_final.isnull().sum())
print("\nSample:")
print(macro_final.tail(10))


Downloaded vix (VIXCLS): 5479 obs
Downloaded tsy10 (DGS10): 5479 obs
Downloaded term_spr (T10Y2Y): 5479 obs
Downloaded ted (TEDRATE): 5234 obs

Final macro shape: (252, 6)

Missing values:
vix          0
vix_ch       1
tsy10        0
tsy10_ch     1
term_spr     0
ted_lvl     11
dtype: int64

Sample:
              vix  vix_ch  tsy10  tsy10_ch  term_spr  ted_lvl
2022-03-31  20.56   -9.59   2.32      0.49      0.04      NaN
2022-04-30  33.40   12.84   2.89      0.57      0.19      NaN
2022-05-31  26.19   -7.21   2.85     -0.04      0.32      NaN
2022-06-30  28.71    2.52   2.98      0.13      0.06      NaN
2022-07-31  21.33   -7.38   2.67     -0.31     -0.22      NaN
2022-08-31  25.87    4.54   3.15      0.48     -0.30      NaN
2022-09-30  31.62    5.75   3.83      0.68     -0.39      NaN
2022-10-31  25.88   -5.74   4.10      0.27     -0.41      NaN
2022-11-30  20.58   -5.30   3.68     -0.42     -0.70      NaN
2022-12-31  21.67    1.09   3.88      0.20     -0.53      NaN


In [20]:
try:
    ebp_raw = fred.get_series('BAMLH0A0HYM2EY', observation_start=start, observation_end=end)
    ebp_raw.index = pd.to_datetime(ebp_raw.index)  # ensure DatetimeIndex
    ebp = ebp_raw.resample('M').last().to_frame(name='ebp_proxy')
    ebp.index.name = 'date'
    ebp = ebp.reset_index()
    ebp['date'] = pd.to_datetime(ebp['date']) + pd.offsets.MonthEnd(0)
    print("EBP proxy downloaded:", ebp.shape)
    print(ebp.tail(5))
except Exception as e:
    print(f"EBP failed: {e}")


EBP proxy downloaded: (0, 2)
Empty DataFrame
Columns: [date, ebp_proxy]
Index: []


In [21]:
import pandas as pd
import numpy as np

# ── Fix 1: Replace TED spread with SOFR spread after 2022 ────────────────────
# TEDRATE was discontinued March 2022
# Fill remaining NaN with the SOFR-based equivalent or forward fill

macro_final['ted_lvl'] = macro_final['ted_lvl'].ffill()

print("Missing after ffill:")
print(macro_final.isnull().sum())

# ── Fix 2: Drop first row NaN from diff() operations ─────────────────────────
macro_final = macro_final.dropna(subset=['vix_ch', 'tsy10_ch'])
print("\nShape after dropping first row NaN:", macro_final.shape)

# ── Fix 3: Fix the date column and alignment ──────────────────────────────────

# Check current date column name
print("\nCurrent columns:", macro_final.columns.tolist())
print("Index name:", macro_final.index.name)

# Reset index properly
macro_final = macro_final.reset_index()

# Rename date column — handle both possible names
if 'DATE' in macro_final.columns:
    macro_final = macro_final.rename(columns={'DATE': 'date'})
elif 'index' in macro_final.columns:
    macro_final = macro_final.rename(columns={'index': 'date'})

# Align to month-end to match bond panel
macro_final['date'] = pd.to_datetime(macro_final['date']) \
                      + pd.offsets.MonthEnd(0)

print("\nFixed date sample:")
print(macro_final[['date', 'vix', 'tsy10', 'term_spr', 'ted_lvl']].tail(10))
print("\nDate range:", macro_final['date'].min(), "to", macro_final['date'].max())
print("Shape:", macro_final.shape)

# ── Fix 4: Download EBP properly ─────────────────────────────────────────────
# Try alternative FRED series for credit conditions

import pandas_datareader.data as web

start = '2002-01-01'
end   = '2022-12-31'

# BAA-AAA spread as proxy for excess bond premium
try:
    baa = web.DataReader('BAA', 'fred', start, end)
    aaa = web.DataReader('AAA', 'fred', start, end)
    
    baa = baa.resample('M').last()
    aaa = aaa.resample('M').last()
    
    # Default spread = BAA - AAA (Moody's)
    def_spread = (baa['BAA'] - aaa['AAA']).to_frame('def_spread')
    def_spread = def_spread.reset_index()
    def_spread = def_spread.rename(columns={'DATE': 'date'})
    def_spread['date'] = pd.to_datetime(def_spread['date']) \
                         + pd.offsets.MonthEnd(0)
    
    print("\nDefault spread (BAA-AAA) downloaded:", def_spread.shape)
    print(def_spread.tail(5))
    
except Exception as e:
    print(f"Default spread failed: {e}")

# Also try industrial production
try:
    ip = web.DataReader('INDPRO', 'fred', start, end)
    ip = ip.resample('M').last()
    ip['indpro_gr'] = ip['INDPRO'].pct_change(1) * 100
    ip = ip[['indpro_gr']].reset_index()
    ip = ip.rename(columns={'DATE': 'date'})
    ip['date'] = pd.to_datetime(ip['date']) + pd.offsets.MonthEnd(0)
    ip = ip.dropna()
    print("\nIndustrial production downloaded:", ip.shape)
    print(ip.tail(5))
    
except Exception as e:
    print(f"Industrial production failed: {e}")

Missing after ffill:
vix         0
vix_ch      1
tsy10       0
tsy10_ch    1
term_spr    0
ted_lvl     0
dtype: int64

Shape after dropping first row NaN: (251, 6)

Current columns: ['vix', 'vix_ch', 'tsy10', 'tsy10_ch', 'term_spr', 'ted_lvl']
Index name: None

Fixed date sample:
          date    vix  tsy10  term_spr  ted_lvl
241 2022-03-31  20.56   2.32      0.04     0.09
242 2022-04-30  33.40   2.89      0.19     0.09
243 2022-05-31  26.19   2.85      0.32     0.09
244 2022-06-30  28.71   2.98      0.06     0.09
245 2022-07-31  21.33   2.67     -0.22     0.09
246 2022-08-31  25.87   3.15     -0.30     0.09
247 2022-09-30  31.62   3.83     -0.39     0.09
248 2022-10-31  25.88   4.10     -0.41     0.09
249 2022-11-30  20.58   3.68     -0.70     0.09
250 2022-12-31  21.67   3.88     -0.53     0.09

Date range: 2002-02-28 00:00:00 to 2022-12-31 00:00:00
Shape: (251, 7)

Default spread (BAA-AAA) downloaded: (252, 2)
          date  def_spread
247 2022-08-31        1.08
248 2022-09-30    

In [22]:
import pandas as pd
import numpy as np

# ── Step 1: Combine all macro series into one dataframe ───────────────────────

# macro_final already has: vix, vix_ch, tsy10, tsy10_ch, term_spr, ted_lvl
# def_spread has: date, def_spread
# ip has: date, indpro_gr

macro_combined = macro_final.merge(def_spread, on='date', how='left')
macro_combined = macro_combined.merge(ip, on='date', how='left')

print("Macro combined shape:", macro_combined.shape)
print("Columns:", macro_combined.columns.tolist())
print("\nMissing values:")
print(macro_combined.isnull().sum())
print("\nSample:")
print(macro_combined.tail(5))

# ── Step 2: Define final macro predictor names ────────────────────────────────

MACRO_PREDICTORS = [
    'vix',        # VIX level — aggregate uncertainty
    'vix_ch',     # Monthly change in VIX — uncertainty shock
    'tsy10',      # 10Y Treasury yield — interest rate level
    'tsy10_ch',   # Monthly change in 10Y yield — rate shock
    'term_spr',   # Term spread (10Y-2Y) — yield curve slope
    'ted_lvl',    # TED spread — interbank funding liquidity
    'def_spread', # BAA-AAA default spread — credit conditions (CGM 2001)
    'indpro_gr',  # Industrial production growth — real economy
]

print(f"\nMacro predictors defined: {len(MACRO_PREDICTORS)}")

# ── Step 3: Merge macro into bond panel ───────────────────────────────────────

# Check date format alignment before merging
print("\nBond panel date sample:")
print(df_model['date'].head(5))
print("\nMacro date sample:")
print(macro_combined['date'].head(5))

# Merge on date
df_model_macro = df_model.merge(macro_combined, on='date', how='left')

print("\nShape after merge:", df_model_macro.shape)
print("\nMissing values in macro columns:")
print(df_model_macro[MACRO_PREDICTORS].isnull().sum())

# ── Step 4: Check merge worked correctly ──────────────────────────────────────

# Should have same number of rows as df_model
assert df_model_macro.shape[0] == df_model.shape[0], \
    f"Row count mismatch: {df_model_macro.shape[0]} vs {df_model.shape[0]}"

# Check a single bond has macro values filled in
sample_bond = df_model_macro[df_model_macro['cusip'] == '001031AC7'][
    ['date', '18_spread', 'vix', 'tsy10', 'term_spr', 'def_spread']
].head(10)
print("\nSample bond with macro variables:")
print(sample_bond)

print("\nMerge successful!")

Macro combined shape: (251, 9)
Columns: ['date', 'vix', 'vix_ch', 'tsy10', 'tsy10_ch', 'term_spr', 'ted_lvl', 'def_spread', 'indpro_gr']

Missing values:
date          0
vix           0
vix_ch        0
tsy10         0
tsy10_ch      0
term_spr      0
ted_lvl       0
def_spread    0
indpro_gr     0
dtype: int64

Sample:
          date    vix  vix_ch  tsy10  tsy10_ch  term_spr  ted_lvl  def_spread  \
246 2022-08-31  25.87    4.54   3.15      0.48     -0.30     0.09        1.08   
247 2022-09-30  31.62    5.75   3.83      0.68     -0.39     0.09        1.10   
248 2022-10-31  25.88   -5.74   4.10      0.27     -0.41     0.09        1.16   
249 2022-11-30  20.58   -5.30   3.68     -0.42     -0.70     0.09        1.17   
250 2022-12-31  21.67    1.09   3.88      0.20     -0.53     0.09        1.16   

     indpro_gr  
246  -0.125169  
247   0.193380  
248  -0.038404  
249  -0.294016  
250  -1.177552  

Macro predictors defined: 8

Bond panel date sample:


NameError: name 'df_model' is not defined

## 4. Variable Selection — Defining Predictors

This cell defines the two groups of predictors used throughout the models: **bond-level characteristics** (age, coupon, face value, duration, momentum, rating, skewness, volatility, etc.) and **firm/equity-level characteristics** (book-to-market, asset growth, profitability, market equity, stock momentum, illiquidity, etc.). It then checks which columns are present in the dataset, reports missing-value counts, and confirms the observation count by IG/HY segment.

In [23]:
import pandas as pd
import numpy as np

# Define your core variable sets
dependent_var = '18_spread'

# Bond-level predictors (the numbered ones are bond-specific)
bond_predictors = ['1_age', '2_coupon', '3_faceval', '6_duration', 
                   '12_mom6', '22_rating', '24_skew', '25_mom6mspread',
                   '27_volatility', '28_VaR', '29_vixbeta', 'dts',
                   'swap_spread', 'convexity', 'RATING_NUM']

# Firm/equity predictors (a selection of the most standard ones)
firm_predictors = ['be_me', 'at_me', 'ni_me', 'gp_at', 'ret_12_1',
                   'ret_1_0', 'ivol_capm_21d', 'market_equity',
                   'at_gr1', 'niq_be', 'debt_me', 'leverage',
                   'beta_60m', 'ami_126d', 'zero_trades_21d']

all_predictors = bond_predictors + firm_predictors

# Check these columns exist
print("Bond predictors available:")
available_bond = [c for c in bond_predictors if c in df.columns]
print(available_bond)

print("\nFirm predictors available:")
available_firm = [c for c in firm_predictors if c in df.columns]
print(available_firm)

# Check coverage - how many observations have all predictors
core_cols = [dependent_var] + available_bond + available_firm
df_core = df[['date', 'cusip', 'RATING_NUM', 'ice_idx'] + core_cols].copy()

print("\nShape with core columns:", df_core.shape)
print("\nMissing values per column:")
print(df_core.isnull().sum())

# IG vs HY breakdown
print("\nIG observations:", (df_core['ice_idx'] == 'c0a0').sum())
print("HY observations:", (df_core['ice_idx'] == 'H0A0').sum())

# Date range check
print("\nDate range:", df_core['date'].min(), "to", df_core['date'].max())
print("Number of months:", df_core['date'].nunique())

Bond predictors available:
['1_age', '2_coupon', '3_faceval', '6_duration', '12_mom6', '22_rating', '24_skew', '25_mom6mspread', '27_volatility', '28_VaR', '29_vixbeta', 'dts', 'swap_spread', 'convexity', 'RATING_NUM']

Firm predictors available:
['be_me', 'at_me', 'ni_me', 'gp_at', 'ret_12_1', 'ret_1_0', 'ivol_capm_21d', 'market_equity', 'at_gr1', 'niq_be', 'debt_me', 'beta_60m', 'ami_126d', 'zero_trades_21d']

Shape with core columns: (1102569, 34)

Missing values per column:
date               0
cusip              0
RATING_NUM         0
ice_idx            0
18_spread          0
1_age              0
2_coupon           0
3_faceval          0
6_duration         0
12_mom6            0
22_rating          0
24_skew            0
25_mom6mspread     0
27_volatility      0
28_VaR             0
29_vixbeta         0
dts                0
swap_spread        0
convexity          0
RATING_NUM         0
be_me              0
at_me              0
ni_me              0
gp_at              0
ret_12_1     

In [25]:
import pandas as pd
import numpy as np

# ── Merge macro into the original bond panel ──────────────────────────────────

# Start fresh from the original df
# macro_combined should already exist from earlier cells
# If not, rebuild it:
macro_combined = macro_final.merge(def_spread, on='date', how='left')
macro_combined = macro_combined.merge(ip, on='date', how='left')

print("macro_combined shape:", macro_combined.shape)
print("macro_combined columns:", macro_combined.columns.tolist())
print("macro_combined date sample:", macro_combined['date'].head(3).tolist())

# Check df date format
print("\ndf date sample:", df['date'].head(3).tolist())

# Merge macro into original df
df_model_macro = df.merge(macro_combined, on='date', how='left')

print("\ndf_model_macro shape:", df_model_macro.shape)
print("Macro columns present:", 
      [c for c in ['vix', 'tsy10', 'term_spr', 
                   'ted_lvl', 'def_spread', 'indpro_gr'] 
       if c in df_model_macro.columns])
print("\nMissing in macro cols:")
print(df_model_macro[['vix', 'tsy10', 'term_spr', 
                       'ted_lvl', 'def_spread', 'indpro_gr']].isnull().sum())

macro_combined shape: (251, 9)
macro_combined columns: ['date', 'vix', 'vix_ch', 'tsy10', 'tsy10_ch', 'term_spr', 'ted_lvl', 'def_spread', 'indpro_gr']
macro_combined date sample: [Timestamp('2002-02-28 00:00:00'), Timestamp('2002-03-31 00:00:00'), Timestamp('2002-04-30 00:00:00')]

df date sample: [Timestamp('2002-12-31 00:00:00'), Timestamp('2003-01-31 00:00:00'), Timestamp('2003-02-28 00:00:00')]

df_model_macro shape: (1102569, 389)
Macro columns present: ['vix', 'tsy10', 'term_spr', 'ted_lvl', 'def_spread', 'indpro_gr']

Missing in macro cols:
vix           0
tsy10         0
term_spr      0
ted_lvl       0
def_spread    0
indpro_gr     0
dtype: int64


In [26]:
# Bond predictors
BOND_PREDICTORS = [
    '1_age', '2_coupon', '3_faceval', '6_duration',
    '12_mom6', '22_rating', '24_skew', '25_mom6mspread',
    '27_volatility', '28_VaR', '29_vixbeta',
    'dts', 'swap_spread', 'convexity',
]

# Firm predictors
FIRM_PREDICTORS = [
    'be_me', 'at_me', 'ni_me', 'gp_at',
    'ret_12_1', 'ret_1_0', 'ivol_capm_21d',
    'market_equity', 'at_gr1', 'niq_be',
    'debt_me', 'beta_60m', 'ami_126d', 'zero_trades_21d'
]

# Macro predictors
MACRO_PREDICTORS = [
    'vix', 'vix_ch', 'tsy10', 'tsy10_ch',
    'term_spr', 'ted_lvl', 'def_spread', 'indpro_gr'
]

PREDICTORS_FULL = BOND_PREDICTORS + FIRM_PREDICTORS + MACRO_PREDICTORS

print(f"Bond:  {len(BOND_PREDICTORS)}")
print(f"Firm:  {len(FIRM_PREDICTORS)}")
print(f"Macro: {len(MACRO_PREDICTORS)}")
print(f"Total: {len(PREDICTORS_FULL)}")
print(f"Duplicates: {len(PREDICTORS_FULL) - len(set(PREDICTORS_FULL))}")

# Check availability
unavailable = [c for c in PREDICTORS_FULL 
               if c not in df_model_macro.columns]
print(f"\nUnavailable: {unavailable}")

# Build clean modelling dataframe
keep_cols = ['date', 'cusip', 'ice_idx', 'RATING_NUM',
             '18_spread'] + PREDICTORS_FULL

df_model_macro = df_model_macro[keep_cols].copy()
df_model_macro = df_model_macro.sort_values(
    ['cusip', 'date']
).reset_index(drop=True)

# Create spread change target
df_model_macro['spread_change'] = (
    df_model_macro.groupby('cusip')['18_spread'].shift(-1)
    - df_model_macro['18_spread']
)
df_model_macro = df_model_macro.dropna(subset=['spread_change'])
df_model_macro = df_model_macro.dropna(
    subset=MACRO_PREDICTORS
)
df_model_macro = df_model_macro.sort_values(
    'date'
).reset_index(drop=True)

TARGET_NEW = 'spread_change'

print("\nFinal shape:", df_model_macro.shape)
print("\nMissing values in all predictors:")
missing = df_model_macro[PREDICTORS_FULL].isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "None — clean!")

# Sample check
print("\nSample bond with macro vars:")
print(df_model_macro[
    df_model_macro['cusip'] == '001031AC7'
][['date', 'spread_change', 'vix', 
   'tsy10', 'def_spread']].head(5))

Bond:  14
Firm:  14
Macro: 8
Total: 36
Duplicates: 0

Unavailable: []

Final shape: (1082801, 42)

Missing values in all predictors:
None — clean!

Sample bond with macro vars:
            date  spread_change    vix  tsy10  def_spread
23028 2002-12-31       0.065001  28.62   3.83        1.24
26569 2003-01-31       0.045262  31.17   4.00        1.18
29840 2003-02-28      -0.020747  29.63   3.71        1.11
32868 2003-03-31       0.013833  29.15   3.83        1.06
38357 2003-04-30      -0.008848  21.21   3.89        1.11


## 5. Baseline Model — OLS on Spread Levels (Expanding Window)

This cell establishes the first baseline using Ordinary Least Squares (OLS) regression with the raw spread level (`18_spread`) as the target variable. An expanding-window approach is used: the model trains on all available months before the test month and predicts one month ahead. Out-of-sample R² is computed following the Gu, Kelly & Xiu (2020) convention — where the benchmark is zero rather than the historical mean — alongside RMSE and MAE.

In [27]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error

# ── Settings ──────────────────────────────────────────────────────────────────

TRAIN_END   = pd.Timestamp('2009-06-30')
months      = sorted(df_model_macro['date'].unique())
test_months = [m for m in months if m > TRAIN_END]

print(f"Training months: {sum(1 for m in months if m <= TRAIN_END)}")
print(f"Test months:     {len(test_months)}")
print(f"Predictors:      {len(PREDICTORS_FULL)}")

# ── Expanding window OLS ──────────────────────────────────────────────────────

results_ols_macro = []

for i, test_month in enumerate(test_months):

    train_data = df_model_macro[df_model_macro['date'] < test_month]
    test_data  = df_model_macro[df_model_macro['date'] == test_month]

    if len(test_data) < 10:
        continue

    X_train = train_data[PREDICTORS_FULL].values
    y_train = train_data[TARGET_NEW].values
    X_test  = test_data[PREDICTORS_FULL].values
    y_test  = test_data[TARGET_NEW].values

    ols = LinearRegression()
    ols.fit(X_train, y_train)
    y_pred = ols.predict(X_test)

    results_ols_macro.append({
        'month':  test_month,
        'n_test': len(y_test),
        'y_test': y_test,
        'pred':   y_pred
    })

    if i % 20 == 0:
        print(f"Month {i}/{len(test_months)}...")

print(f"\nDone! Processed {len(results_ols_macro)} test months")

# ── Metrics ───────────────────────────────────────────────────────────────────

all_y    = np.concatenate([r['y_test'] for r in results_ols_macro])
all_pred = np.concatenate([r['pred']   for r in results_ols_macro])

ss_res = np.sum((all_y - all_pred) ** 2)
ss_tot = np.sum(all_y ** 2)
r2_oos = 1 - ss_res / ss_tot
rmse   = np.sqrt(mean_squared_error(all_y, all_pred))
mae    = mean_absolute_error(all_y, all_pred)

print("\n" + "="*50)
print("OLS — BOND + FIRM + MACRO (36 predictors)")
print("="*50)
print(f"OOS R²: {r2_oos:.4f}")
print(f"RMSE:   {rmse:.4f}")
print(f"MAE:    {mae:.4f}")
print(f"N obs:  {len(all_y):,}")

# ── Compare with previous OLS (28 predictors) ─────────────────────────────────

print("\n" + "="*50)
print("COMPARISON")
print("="*50)
print(f"{'Model':<30} {'OOS R²':>8}")
print("-"*50)
print(f"{'OLS (28 bond/firm only)':<30} {'0.0397':>8}")
print(f"{'OLS (36 with macro)':<30} {r2_oos:>8.4f}")
print(f"{'Difference':<30} {r2_oos - 0.0397:>+8.4f}")

c:\Users\vishn\anaconda3\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.3
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


Training months: 84
Test months:     160
Predictors:      36
Month 0/160...
Month 20/160...
Month 40/160...
Month 60/160...
Month 80/160...
Month 100/160...
Month 120/160...
Month 140/160...

Done! Processed 160 test months

OLS — BOND + FIRM + MACRO (36 predictors)
OOS R²: 0.0398
RMSE:   0.1175
MAE:    0.0688
N obs:  796,546

COMPARISON
Model                            OOS R²
--------------------------------------------------
OLS (28 bond/firm only)          0.0397
OLS (36 with macro)              0.0398
Difference                      +0.0001


## 7. Penalised Linear Models — Ridge and LASSO

This cell extends the linear framework with regularisation. **Ridge** applies L2 penalisation (shrinks all coefficients towards zero without eliminating any); **LASSO** applies L1 penalisation (performs automatic variable selection by zeroing some coefficients entirely). For each test month, both models select their penalty parameter (alpha) by minimising MSE on the most recent 24 months of training data, then refit on the full training window before predicting.

In [28]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler

# ── Results storage ───────────────────────────────────────────────────────────

results_all = {
    'ols':   [],
    'ridge': [],
    'lasso': []
}

# ── Hyperparameters to try (tuned on last 2 years of training data) ───────────

RIDGE_ALPHAS = [0.001, 0.01, 0.1, 1.0, 10.0]
LASSO_ALPHAS = [0.0001, 0.001, 0.01, 0.1]

# ── Expanding window ──────────────────────────────────────────────────────────

for i, test_month in enumerate(test_months):
    
    train_data = df_model_macro[df_model_macro['date'] < test_month]
    test_data  = df_model_macro[df_model_macro['date'] == test_month]
    
    if len(test_data) < 10:
        continue

    # Use last 24 months of training data as validation for tuning
    all_train_months = sorted(train_data['date'].unique())
    if len(all_train_months) > 24:
        val_months  = all_train_months[-24:]
        pure_train  = train_data[~train_data['date'].isin(val_months)]
        val_data    = train_data[train_data['date'].isin(val_months)]
    else:
        pure_train = train_data
        val_data   = train_data

    X_pure  = pure_train[PREDICTORS_FULL].values
    y_pure  = pure_train[TARGET_NEW].values
    X_val   = val_data[PREDICTORS_FULL].values
    y_val   = val_data[TARGET_NEW].values
    X_train_full = train_data[PREDICTORS_FULL].values
    y_train_full = train_data[TARGET_NEW].values
    X_test  = test_data[PREDICTORS_FULL].values
    y_test  = test_data[TARGET_NEW].values

    # ── OLS ───────────────────────────────────────────────────────────────────
    ols = LinearRegression()
    ols.fit(X_train_full, y_train_full)
    pred_ols = ols.predict(X_test)

    # ── Ridge — tune alpha on validation set ──────────────────────────────────
    best_ridge_alpha = RIDGE_ALPHAS[0]
    best_ridge_mse   = np.inf
    for alpha in RIDGE_ALPHAS:
        m = Ridge(alpha=alpha)
        m.fit(X_pure, y_pure)
        mse = mean_squared_error(y_val, m.predict(X_val))
        if mse < best_ridge_mse:
            best_ridge_mse   = mse
            best_ridge_alpha = alpha
    ridge_final = Ridge(alpha=best_ridge_alpha)
    ridge_final.fit(X_train_full, y_train_full)
    pred_ridge = ridge_final.predict(X_test)

    # ── LASSO — tune alpha on validation set ──────────────────────────────────
    best_lasso_alpha = LASSO_ALPHAS[0]
    best_lasso_mse   = np.inf
    for alpha in LASSO_ALPHAS:
        m = Lasso(alpha=alpha, max_iter=5000)
        m.fit(X_pure, y_pure)
        mse = mean_squared_error(y_val, m.predict(X_val))
        if mse < best_lasso_mse:
            best_lasso_mse   = mse
            best_lasso_alpha = alpha
    lasso_final = Lasso(alpha=best_lasso_alpha, max_iter=5000)
    lasso_final.fit(X_train_full, y_train_full)
    pred_lasso = lasso_final.predict(X_test)

    # ── Store ─────────────────────────────────────────────────────────────────
    for model, pred in [('ols', pred_ols), 
                        ('ridge', pred_ridge), 
                        ('lasso', pred_lasso)]:
        results_all[model].append({
            'month':  test_month,
            'y_test': y_test,
            'pred':   pred
        })

    if i % 20 == 0:
        print(f"Processed {i}/{len(test_months)} months...")

print("Done!")

# ── Compute OOS R² for all models ─────────────────────────────────────────────

print("\n" + "="*50)
print("RESULTS SUMMARY")
print("="*50)

for model_name, res_list in results_all.items():
    all_y    = np.concatenate([r['y_test'] for r in res_list])
    all_pred = np.concatenate([r['pred']   for r in res_list])
    
    ss_res = np.sum((all_y - all_pred) ** 2)
    ss_tot = np.sum(all_y ** 2)
    r2_oos = 1 - ss_res / ss_tot
    rmse   = np.sqrt(mean_squared_error(all_y, all_pred))
    mae    = mean_absolute_error(all_y, all_pred)
    
    print(f"\n{model_name.upper()}")
    print(f"  OOS R²: {r2_oos:.4f}")
    print(f"  RMSE:   {rmse:.4f}")
    print(f"  MAE:    {mae:.4f}")

Processed 0/160 months...
Processed 20/160 months...
Processed 40/160 months...
Processed 60/160 months...
Processed 80/160 months...
Processed 100/160 months...
Processed 120/160 months...
Processed 140/160 months...
Done!

RESULTS SUMMARY

OLS
  OOS R²: 0.0398
  RMSE:   0.1175
  MAE:    0.0688

RIDGE
  OOS R²: 0.0398
  RMSE:   0.1175
  MAE:    0.0688

LASSO
  OOS R²: 0.0381
  RMSE:   0.1176
  MAE:    0.0684


In [29]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error

# ── Hyperparameter grids ──────────────────────────────────────────────────────

RIDGE_ALPHAS = [0.001, 0.01, 0.1, 1.0, 10.0]
LASSO_ALPHAS = [0.0001, 0.001, 0.01, 0.1]
RANDOM_STATE = 42
MAX_TRAIN_OBS = 50000

# ── Storage ───────────────────────────────────────────────────────────────────

results_ridge_macro = []
results_lasso_macro = []

# ── Expanding window ──────────────────────────────────────────────────────────

for i, test_month in enumerate(test_months):

    train_data = df_model_macro[df_model_macro['date'] < test_month]
    test_data  = df_model_macro[df_model_macro['date'] == test_month]

    if len(test_data) < 10:
        continue

    # Subsample training data
    if len(train_data) > MAX_TRAIN_OBS:
        train_sampled = train_data.sample(
            n=MAX_TRAIN_OBS, random_state=RANDOM_STATE
        )
    else:
        train_sampled = train_data

    # Validation split — last 24 months of training
    all_train_months = sorted(
        df_model_macro[
            df_model_macro['date'] < test_month
        ]['date'].unique()
    )
    if len(all_train_months) > 24:
        val_months = all_train_months[-24:]
        pure_train = train_sampled[
            ~train_sampled['date'].isin(val_months)
        ]
        val_data   = train_sampled[
            train_sampled['date'].isin(val_months)
        ]
    else:
        pure_train = train_sampled
        val_data   = train_sampled

    if len(val_data) == 0:
        val_data = pure_train

    # Arrays
    X_pure       = pure_train[PREDICTORS_FULL].values
    y_pure       = pure_train[TARGET_NEW].values
    X_val        = val_data[PREDICTORS_FULL].values
    y_val        = val_data[TARGET_NEW].values
    X_train_full = train_sampled[PREDICTORS_FULL].values
    y_train_full = train_sampled[TARGET_NEW].values
    X_test       = test_data[PREDICTORS_FULL].values
    y_test       = test_data[TARGET_NEW].values

    # ── Ridge ─────────────────────────────────────────────────────────────────
    best_ridge_alpha = RIDGE_ALPHAS[0]
    best_ridge_mse   = np.inf
    for alpha in RIDGE_ALPHAS:
        m = Ridge(alpha=alpha)
        m.fit(X_pure, y_pure)
        mse = mean_squared_error(y_val, m.predict(X_val))
        if mse < best_ridge_mse:
            best_ridge_mse   = mse
            best_ridge_alpha = alpha

    ridge_final = Ridge(alpha=best_ridge_alpha)
    ridge_final.fit(X_train_full, y_train_full)

    results_ridge_macro.append({
        'month':      test_month,
        'y_test':     y_test,
        'pred':       ridge_final.predict(X_test),
        'best_alpha': best_ridge_alpha
    })

    # ── LASSO ─────────────────────────────────────────────────────────────────
    best_lasso_alpha = LASSO_ALPHAS[0]
    best_lasso_mse   = np.inf
    for alpha in LASSO_ALPHAS:
        m = Lasso(alpha=alpha, max_iter=5000)
        m.fit(X_pure, y_pure)
        mse = mean_squared_error(y_val, m.predict(X_val))
        if mse < best_lasso_mse:
            best_lasso_mse   = mse
            best_lasso_alpha = alpha

    lasso_final = Lasso(alpha=best_lasso_alpha, max_iter=5000)
    lasso_final.fit(X_train_full, y_train_full)

    results_lasso_macro.append({
        'month':      test_month,
        'y_test':     y_test,
        'pred':       lasso_final.predict(X_test),
        'best_alpha': best_lasso_alpha
    })

    if i % 20 == 0:
        print(f"Month {i}/{len(test_months)} — "
              f"Ridge α={best_ridge_alpha}, "
              f"LASSO α={best_lasso_alpha}")

print(f"\nDone!")

# ── Metrics ───────────────────────────────────────────────────────────────────

def compute_metrics(res_list):
    all_y    = np.concatenate([r['y_test'] for r in res_list])
    all_pred = np.concatenate([r['pred']   for r in res_list])
    ss_res   = np.sum((all_y - all_pred) ** 2)
    ss_tot   = np.sum(all_y ** 2)
    return {
        'r2':   round(1 - ss_res / ss_tot, 4),
        'rmse': round(np.sqrt(mean_squared_error(all_y, all_pred)), 4),
        'mae':  round(mean_absolute_error(all_y, all_pred), 4),
        'n':    len(all_y)
    }

ridge_m = compute_metrics(results_ridge_macro)
lasso_m = compute_metrics(results_lasso_macro)

print("\n" + "="*65)
print("LINEAR MODELS — BOND + FIRM + MACRO (36 predictors)")
print("="*65)
print(f"{'Model':<12} {'OOS R²':>8} {'RMSE':>8} {'MAE':>8} {'N obs':>10}")
print("-"*65)
print(f"{'OLS':<12} {r2_oos:>8.4f} {rmse:>8.4f} {mae:>8.4f} {len(all_y):>10,}")
print(f"{'Ridge':<12} {ridge_m['r2']:>8.4f} {ridge_m['rmse']:>8.4f} {ridge_m['mae']:>8.4f} {ridge_m['n']:>10,}")
print(f"{'LASSO':<12} {lasso_m['r2']:>8.4f} {lasso_m['rmse']:>8.4f} {lasso_m['mae']:>8.4f} {lasso_m['n']:>10,}")

print("\n" + "="*65)
print("COMPARISON WITH PREVIOUS (28 predictors)")
print("="*65)
print(f"{'Model':<12} {'Old R²':>8} {'New R²':>8} {'Change':>8}")
print("-"*65)
print(f"{'OLS':<12} {'0.0397':>8} {r2_oos:>8.4f} {r2_oos-0.0397:>+8.4f}")
print(f"{'Ridge':<12} {'0.0397':>8} {ridge_m['r2']:>8.4f} {ridge_m['r2']-0.0397:>+8.4f}")
print(f"{'LASSO':<12} {'0.0383':>8} {lasso_m['r2']:>8.4f} {lasso_m['r2']-0.0383:>+8.4f}")

# ── Alpha selection frequency ─────────────────────────────────────────────────

from collections import Counter

ridge_alphas = [r['best_alpha'] for r in results_ridge_macro]
lasso_alphas = [r['best_alpha'] for r in results_lasso_macro]

print("\nRidge alpha selection:")
for alpha, count in sorted(Counter(ridge_alphas).items()):
    print(f"  α={alpha}: {count} months")

print("\nLASSO alpha selection:")
for alpha, count in sorted(Counter(lasso_alphas).items()):
    print(f"  α={alpha}: {count} months")

Month 0/160 — Ridge α=10.0, LASSO α=0.001
Month 20/160 — Ridge α=10.0, LASSO α=0.0001
Month 40/160 — Ridge α=10.0, LASSO α=0.0001
Month 60/160 — Ridge α=10.0, LASSO α=0.001
Month 80/160 — Ridge α=10.0, LASSO α=0.001
Month 100/160 — Ridge α=10.0, LASSO α=0.0001
Month 120/160 — Ridge α=0.001, LASSO α=0.0001
Month 140/160 — Ridge α=0.001, LASSO α=0.0001

Done!

LINEAR MODELS — BOND + FIRM + MACRO (36 predictors)
Model          OOS R²     RMSE      MAE      N obs
-----------------------------------------------------------------
OLS            0.0381   0.1176   0.0684    796,546
Ridge          0.0389   0.1176   0.0688    796,546
LASSO          0.0352   0.1178   0.0685    796,546

COMPARISON WITH PREVIOUS (28 predictors)
Model          Old R²   New R²   Change
-----------------------------------------------------------------
OLS            0.0397   0.0381  -0.0016
Ridge          0.0397   0.0389  -0.0008
LASSO          0.0383   0.0352  -0.0031

Ridge alpha selection:
  α=0.001: 30 months
  α=

## 8. Non-linear Model — Random Forest (Walk-forward)

This cell trains a Random Forest under the same expanding-window scheme. Because Random Forests are computationally expensive on large panels, training data is capped at 50,000 observations per month via random subsampling. Hyperparameters (tree depth and number of candidate features per split) are selected using the most recent 24 months as a held-out validation set. Feature importances are stored each month and averaged at the end to identify which predictors drive performance.

In [30]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

# ── Settings ──────────────────────────────────────────────────────────────────

MAX_TRAIN_OBS = 50000   # cap training observations per month
N_TREES       = 100     # reduce from 300 to 100 for speed
RANDOM_STATE  = 42

RF_PARAMS = [
    {'max_depth': 2, 'max_features': 5},
    {'max_depth': 3, 'max_features': 5},
    {'max_depth': 3, 'max_features': 10},
    {'max_depth': 4, 'max_features': 10},
]

results_rf = []

for i, test_month in enumerate(test_months):

    train_data = df_model_macro[df_model_macro['date'] < test_month]
    test_data  = df_model_macro[df_model_macro['date'] == test_month]

    if len(test_data) < 10:
        continue

    # ── Subsample training data if too large ──────────────────────────────────
    if len(train_data) > MAX_TRAIN_OBS:
        train_data = train_data.sample(
            n=MAX_TRAIN_OBS, 
            random_state=RANDOM_STATE
        )

    # ── Validation split — last 24 months ────────────────────────────────────
    all_train_months = sorted(
        df_model_macro[df_model_macro['date'] < test_month]['date'].unique()
    )
    if len(all_train_months) > 24:
        val_months = all_train_months[-24:]
        pure_train = train_data[~train_data['date'].isin(val_months)]
        val_data   = train_data[train_data['date'].isin(val_months)]
    else:
        pure_train = train_data
        val_data   = train_data

    # Ensure val data is not empty
    if len(val_data) == 0:
        val_data   = pure_train
        pure_train = train_data

    X_pure       = pure_train[PREDICTORS_FULL].values
    y_pure       = pure_train[TARGET_NEW].values
    X_val        = val_data[PREDICTORS_FULL].values
    y_val        = val_data[TARGET_NEW].values
    X_train_full = train_data[PREDICTORS_FULL].values
    y_train_full = train_data[TARGET_NEW].values
    X_test       = test_data[PREDICTORS_FULL].values
    y_test       = test_data[TARGET_NEW].values

    # ── Tune on validation ────────────────────────────────────────────────────
    best_params  = RF_PARAMS[0]
    best_val_mse = np.inf

    for params in RF_PARAMS:
        rf = RandomForestRegressor(
            n_estimators=N_TREES,
            max_depth=params['max_depth'],
            max_features=params['max_features'],
            n_jobs=-1,
            random_state=RANDOM_STATE
        )
        rf.fit(X_pure, y_pure)
        val_mse = mean_squared_error(y_val, rf.predict(X_val))
        if val_mse < best_val_mse:
            best_val_mse = val_mse
            best_params  = params

    # ── Final model on full (subsampled) training data ────────────────────────
    rf_final = RandomForestRegressor(
        n_estimators=N_TREES,
        max_depth=best_params['max_depth'],
        max_features=best_params['max_features'],
        n_jobs=-1,
        random_state=RANDOM_STATE
    )
    rf_final.fit(X_train_full, y_train_full)
    pred_rf = rf_final.predict(X_test)

    results_rf.append({
        'month':              test_month,
        'y_test':             y_test,
        'pred':               pred_rf,
        'feature_importance': rf_final.feature_importances_,
        'best_depth':         best_params['max_depth'],
        'best_features':      best_params['max_features']
    })

    if i % 10 == 0:
        elapsed_months = i + 1
        print(f"Month {i}/{len(test_months)} — "
              f"depth={best_params['max_depth']}, "
              f"features={best_params['max_features']}, "
              f"train_n={len(train_data):,}, "
              f"test_n={len(test_data):,}")

print("\nDone!")

# ── OOS R² ────────────────────────────────────────────────────────────────────

all_y    = np.concatenate([r['y_test'] for r in results_rf])
all_pred = np.concatenate([r['pred']   for r in results_rf])

ss_res = np.sum((all_y - all_pred) ** 2)
ss_tot = np.sum(all_y ** 2)
r2_oos = 1 - ss_res / ss_tot
rmse   = np.sqrt(mean_squared_error(all_y, all_pred))
mae    = mean_absolute_error(all_y, all_pred)

print("\n" + "="*50)
print("RANDOM FOREST RESULTS")
print("="*50)
print(f"OOS R²: {r2_oos:.4f}")
print(f"RMSE:   {rmse:.4f}")
print(f"MAE:    {mae:.4f}")

# ── Variable importance ───────────────────────────────────────────────────────
# Filter to results that have the right length
n_features = len(PREDICTORS_FULL)
valid_importances = [
    r['feature_importance'] for r in results_rf 
    if len(r['feature_importance']) == n_features
]

print(f"Valid importance arrays: {len(valid_importances)} out of {len(results_rf)}")

avg_importance = np.mean(valid_importances, axis=0)

importance_df = pd.DataFrame({
    'predictor':  PREDICTORS_FULL,
    'importance': avg_importance
}).sort_values('importance', ascending=False)

print("\nTop 15 most important predictors:")
print(importance_df.head(15).to_string(index=False))

print("\nBottom 10 least important predictors:")
print(importance_df.tail(10).to_string(index=False))


Month 0/160 — depth=4, features=10, train_n=50,000, test_n=3,308
Month 10/160 — depth=4, features=10, train_n=50,000, test_n=3,474
Month 20/160 — depth=4, features=10, train_n=50,000, test_n=3,749
Month 30/160 — depth=3, features=5, train_n=50,000, test_n=4,004
Month 40/160 — depth=4, features=10, train_n=50,000, test_n=4,434
Month 50/160 — depth=4, features=10, train_n=50,000, test_n=4,736
Month 60/160 — depth=4, features=10, train_n=50,000, test_n=5,073
Month 70/160 — depth=4, features=10, train_n=50,000, test_n=5,298
Month 80/160 — depth=4, features=10, train_n=50,000, test_n=5,519
Month 90/160 — depth=3, features=10, train_n=50,000, test_n=5,402
Month 100/160 — depth=4, features=10, train_n=50,000, test_n=5,449
Month 110/160 — depth=4, features=10, train_n=50,000, test_n=5,461
Month 120/160 — depth=4, features=10, train_n=50,000, test_n=5,440
Month 130/160 — depth=4, features=10, train_n=50,000, test_n=5,705
Month 140/160 — depth=4, features=10, train_n=50,000, test_n=5,843
Month 1

## 9. Fix — Clean Predictor List and Rebuild Model DataFrame

This cell resolves the mismatch identified above by redefining `PREDICTORS` as exactly 28 unique features with no duplicates, and rebuilding `df_model` from scratch to ensure full consistency. A quick sanity-check RF fit on the final test month confirms that feature dimensions now align correctly.

In [13]:
# ── Step 1: Redefine clean PREDICTORS (28 features, no duplicates) ────────────

PREDICTORS = [
    '1_age', '2_coupon', '3_faceval', '6_duration',
    '12_mom6', '22_rating', '24_skew', '25_mom6mspread',
    '27_volatility', '28_VaR', '29_vixbeta',
    'dts', 'swap_spread', 'convexity',
    'be_me', 'at_me', 'ni_me', 'gp_at',
    'ret_12_1', 'ret_1_0', 'ivol_capm_21d',
    'market_equity', 'at_gr1', 'niq_be',
    'debt_me', 'beta_60m', 'ami_126d', 'zero_trades_21d'
]

TARGET_NEW = 'spread_change'

# ── Step 2: Rebuild df_model cleanly from the original df ─────────────────────

keep_cols = ['date', 'cusip', 'ice_idx', 'RATING_NUM', '18_spread'] + PREDICTORS

df_model = df[keep_cols].copy()
df_model = df_model.sort_values(['cusip', 'date']).reset_index(drop=True)

# Create spread change target
df_model['spread_change'] = df_model.groupby('cusip')['18_spread'].shift(-1) - df_model['18_spread']
df_model = df_model.dropna(subset=['spread_change'])
df_model = df_model.sort_values('date').reset_index(drop=True)

print("df_model shape:", df_model.shape)
print("Columns:", df_model.columns.tolist())
print("PREDICTORS length:", len(PREDICTORS))

# ── Step 3: Verify no column mismatch ─────────────────────────────────────────

months = sorted(df_model['date'].unique())
TRAIN_END = pd.Timestamp('2009-06-30')
test_months = [m for m in months if m > TRAIN_END]

# Quick test on one month
test_month = test_months[-1]
train_data = df_model[df_model['date'] < test_month].sample(n=1000, random_state=42)
test_data  = df_model[df_model['date'] == test_month]

X_train = train_data[PREDICTORS].values
X_test  = test_data[PREDICTORS].values

print("\nX_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Feature count matches PREDICTORS:", X_train.shape[1] == len(PREDICTORS))

# ── Step 4: Fit RF and get feature importance ─────────────────────────────────

from sklearn.ensemble import RandomForestRegressor

train_data = df_model[df_model['date'] < test_month]
if len(train_data) > 50000:
    train_data = train_data.sample(n=50000, random_state=42)

X_train = train_data[PREDICTORS].values
y_train = train_data[TARGET_NEW].values
X_test  = test_data[PREDICTORS].values
y_test  = test_data[TARGET_NEW].values

rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=4,
    max_features=10,
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train, y_train)

importance_df = pd.DataFrame({
    'predictor':  PREDICTORS,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 15 most important predictors (RF):")
print(importance_df.head(15).to_string(index=False))

print("\nBottom 10 least important:")
print(importance_df.tail(10).to_string(index=False))

print(f"\nImportances sum to: {rf.feature_importances_.sum():.4f}")

df_model shape: (1082801, 34)
Columns: ['date', 'cusip', 'ice_idx', 'RATING_NUM', '18_spread', '1_age', '2_coupon', '3_faceval', '6_duration', '12_mom6', '22_rating', '24_skew', '25_mom6mspread', '27_volatility', '28_VaR', '29_vixbeta', 'dts', 'swap_spread', 'convexity', 'be_me', 'at_me', 'ni_me', 'gp_at', 'ret_12_1', 'ret_1_0', 'ivol_capm_21d', 'market_equity', 'at_gr1', 'niq_be', 'debt_me', 'beta_60m', 'ami_126d', 'zero_trades_21d', 'spread_change']
PREDICTORS length: 28

X_train shape: (1000, 28)
X_test shape: (5660, 28)
Feature count matches PREDICTORS: True

Top 15 most important predictors (RF):
     predictor  importance
25_mom6mspread    0.259287
    6_duration    0.157320
           dts    0.092525
   swap_spread    0.082226
     convexity    0.057535
 market_equity    0.044961
     22_rating    0.042948
      ami_126d    0.035987
       ret_1_0    0.021878
      2_coupon    0.021325
         1_age    0.017925
 27_volatility    0.016279
         ni_me    0.016135
         be_m

## Setup: Install XGBoost

Installs the XGBoost library, which will be used alongside Random Forests in the next cell for the tree-ensemble comparison.

In [34]:
%pip install xgboost

  Using cached numpy-1.22.4-cp39-cp39-win_amd64.whl (14.7 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.24.3
    Uninstalling numpy-1.24.3:
      Successfully uninstalled numpy-1.24.3
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\vishn\\anaconda3\\Lib\\site-packages\\~%mpy\\.libs\\libopenblas64__v0.3.21-gcc_10_3_0.dll'
Consider using the `--user` option or check the permissions.



## 10. Tree Ensemble Models — Random Forest vs XGBoost (Walk-forward)

This is the main model comparison cell. Both Random Forest and XGBoost are trained under the same expanding-window walk-forward protocol with validation-based hyperparameter selection. Training data is capped at 50,000 observations per month. At the end, out-of-sample R², RMSE, and MAE are reported for all five models (OLS, Ridge, LASSO, RF, XGBoost), and average feature importances across months are compared between RF and XGBoost.

In [35]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error
from collections import Counter

# ── Settings ──────────────────────────────────────────────────────────────────

MAX_TRAIN_OBS = 50000
RANDOM_STATE  = 42

XGB_PARAMS = [
    {'max_depth': 2, 'learning_rate': 0.01, 'n_estimators': 200},
    {'max_depth': 3, 'learning_rate': 0.01, 'n_estimators': 200},
    {'max_depth': 2, 'learning_rate': 0.05, 'n_estimators': 200},
    {'max_depth': 3, 'learning_rate': 0.05, 'n_estimators': 200},
]

# ── Changed: results_xgb → results_xgb_macro ─────────────────────────────────
results_xgb_macro = []

for i, test_month in enumerate(test_months):

    # ── Changed: df_model → df_model_macro ───────────────────────────────────
    train_data = df_model_macro[df_model_macro['date'] < test_month]
    test_data  = df_model_macro[df_model_macro['date'] == test_month]

    if len(test_data) < 10:
        continue

    if len(train_data) > MAX_TRAIN_OBS:
        train_sampled = train_data.sample(
            n=MAX_TRAIN_OBS, random_state=RANDOM_STATE
        )
    else:
        train_sampled = train_data

    # ── Changed: df_model → df_model_macro ───────────────────────────────────
    all_train_months = sorted(
        df_model_macro[
            df_model_macro['date'] < test_month
        ]['date'].unique()
    )
    if len(all_train_months) > 24:
        val_months = all_train_months[-24:]
        pure_train = train_sampled[
            ~train_sampled['date'].isin(val_months)
        ]
        val_data   = train_sampled[
            train_sampled['date'].isin(val_months)
        ]
    else:
        pure_train = train_sampled
        val_data   = train_sampled

    if len(val_data) == 0:
        val_data = pure_train

    # ── Changed: PREDICTORS → PREDICTORS_FULL ────────────────────────────────
    X_pure       = pure_train[PREDICTORS_FULL].values
    y_pure       = pure_train[TARGET_NEW].values
    X_val        = val_data[PREDICTORS_FULL].values
    y_val        = val_data[TARGET_NEW].values
    X_train_full = train_sampled[PREDICTORS_FULL].values
    y_train_full = train_sampled[TARGET_NEW].values
    X_test       = test_data[PREDICTORS_FULL].values
    y_test       = test_data[TARGET_NEW].values

    # ── Tune on validation ────────────────────────────────────────────────────
    best_xgb_params = XGB_PARAMS[0]
    best_xgb_mse    = np.inf

    for params in XGB_PARAMS:
        m = xgb.XGBRegressor(
            max_depth=params['max_depth'],
            learning_rate=params['learning_rate'],
            n_estimators=params['n_estimators'],
            subsample=0.8,
            colsample_bytree=0.8,
            n_jobs=-1,
            random_state=RANDOM_STATE,
            verbosity=0
        )
        m.fit(X_pure, y_pure,
              eval_set=[(X_val, y_val)],
              verbose=False)
        mse = mean_squared_error(y_val, m.predict(X_val))
        if mse < best_xgb_mse:
            best_xgb_mse    = mse
            best_xgb_params = params

    xgb_final = xgb.XGBRegressor(
        max_depth=best_xgb_params['max_depth'],
        learning_rate=best_xgb_params['learning_rate'],
        n_estimators=best_xgb_params['n_estimators'],
        subsample=0.8,
        colsample_bytree=0.8,
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbosity=0
    )
    xgb_final.fit(X_train_full, y_train_full)

    # ── Changed: results_xgb → results_xgb_macro ─────────────────────────────
    results_xgb_macro.append({
        'month':              test_month,
        'y_test':             y_test,
        'pred':               xgb_final.predict(X_test),
        'feature_importance': xgb_final.feature_importances_.tolist(),
        'best_depth':         best_xgb_params['max_depth'],
        'best_lr':            best_xgb_params['learning_rate']
    })

    if i % 10 == 0:
        print(f"Month {i}/{len(test_months)} — "
              f"XGB depth={best_xgb_params['max_depth']}, "
              f"lr={best_xgb_params['learning_rate']}, "
              f"test_n={len(test_data):,}")

print("\nDone!")

# ── Metrics ───────────────────────────────────────────────────────────────────

all_y_xgb    = np.concatenate([r['y_test'] for r in results_xgb_macro])
all_pred_xgb = np.concatenate([r['pred']   for r in results_xgb_macro])

ss_res  = np.sum((all_y_xgb - all_pred_xgb) ** 2)
ss_tot  = np.sum(all_y_xgb ** 2)
r2_xgb  = 1 - ss_res / ss_tot
rmse_xgb = np.sqrt(mean_squared_error(all_y_xgb, all_pred_xgb))
mae_xgb  = mean_absolute_error(all_y_xgb, all_pred_xgb)

print("\n" + "="*55)
print("XGBOOST — BOND + FIRM + MACRO (36 predictors)")
print("="*55)
print(f"OOS R²: {r2_xgb:.4f}")
print(f"RMSE:   {rmse_xgb:.4f}")
print(f"MAE:    {mae_xgb:.4f}")
print(f"N obs:  {len(all_y_xgb):,}")

# ── Hyperparameter selection ──────────────────────────────────────────────────

depths = [r['best_depth'] for r in results_xgb_macro]
lrs    = [r['best_lr']    for r in results_xgb_macro]

print("\nDepth selection:")
for d, count in sorted(Counter(depths).items()):
    print(f"  depth={d}: {count} months")

print("\nLearning rate selection:")
for lr, count in sorted(Counter(lrs).items()):
    print(f"  lr={lr}: {count} months")

# ── Feature importance ────────────────────────────────────────────────────────

xgb_imp = np.mean(
    [r['feature_importance'] for r in results_xgb_macro], axis=0
)

importance_df = pd.DataFrame({
    'predictor':  PREDICTORS_FULL,
    'importance': xgb_imp,
    'category':   (['Bond']  * len(BOND_PREDICTORS) +
                   ['Firm']  * len(FIRM_PREDICTORS) +
                   ['Macro'] * len(MACRO_PREDICTORS))
}).sort_values('importance', ascending=False)

print("\nTop 20 most important predictors (XGBoost):")
print(importance_df.head(20).to_string(index=False))

print("\nMacro predictors specifically:")
print(importance_df[
    importance_df['category'] == 'Macro'
].to_string(index=False))

# ── Full comparison table ─────────────────────────────────────────────────────

print("\n" + "="*70)
print("COMPLETE RESULTS — ALL MODELS, ALL PREDICTOR SETS")
print("="*70)
print(f"{'Model':<25} {'OOS R²':>8} {'RMSE':>8} {'MAE':>8}")
print("-"*70)

# Generation 2 results (28 predictors)
for name, r2, rmse, mae in [
    ('OLS (28 vars)',     0.0397, 0.1175, 0.0689),
    ('Ridge (28 vars)',   0.0397, 0.1175, 0.0689),
    ('LASSO (28 vars)',   0.0383, 0.1176, 0.0685),
    ('RF (28 vars)',      0.0430, 0.1173, 0.0681),
    ('XGBoost (28 vars)', 0.0718, 0.1156, 0.0680),
]:
    print(f"{name:<25} {r2:>8.4f} {rmse:>8.4f} {mae:>8.4f}")

print("-"*70)

# Generation 3 results (36 predictors) — paste OLS/Ridge/LASSO from earlier
for name, r2, rmse, mae in [
    ('OLS (36 vars)',   0.0398,  0.1175,  0.0688),
    ('Ridge (36 vars)', 0.0389,  0.1176,  0.0688),
    ('LASSO (36 vars)', 0.0352,  0.1178,  0.0685),
    ('RF (36 vars)',    r2_rf,   rmse_rf,   mae_rf),
]:
    print(f"{name:<25} {r2:>8.4f} {rmse:>8.4f} {mae:>8.4f}")

print(f"{'XGBoost (36 vars)':<25} {r2_xgb:>8.4f} "
      f"{rmse_xgb:>8.4f} {mae_xgb:>8.4f}")

Month 0/160 — XGB depth=2, lr=0.05, test_n=3,308
Month 10/160 — XGB depth=3, lr=0.05, test_n=3,474
Month 20/160 — XGB depth=3, lr=0.01, test_n=3,749
Month 30/160 — XGB depth=2, lr=0.05, test_n=4,004
Month 40/160 — XGB depth=3, lr=0.05, test_n=4,434
Month 50/160 — XGB depth=3, lr=0.05, test_n=4,736
Month 60/160 — XGB depth=3, lr=0.01, test_n=5,073
Month 70/160 — XGB depth=3, lr=0.05, test_n=5,298
Month 80/160 — XGB depth=2, lr=0.05, test_n=5,519
Month 90/160 — XGB depth=3, lr=0.05, test_n=5,402
Month 100/160 — XGB depth=3, lr=0.05, test_n=5,449
Month 110/160 — XGB depth=2, lr=0.05, test_n=5,461
Month 120/160 — XGB depth=3, lr=0.05, test_n=5,440
Month 130/160 — XGB depth=2, lr=0.05, test_n=5,705
Month 140/160 — XGB depth=3, lr=0.05, test_n=5,843
Month 150/160 — XGB depth=3, lr=0.05, test_n=5,851

Done!

XGBOOST — BOND + FIRM + MACRO (36 predictors)
OOS R²: 0.0743
RMSE:   0.1154
MAE:    0.0680
N obs:  796,546

Depth selection:
  depth=2: 51 months
  depth=3: 109 months

Learning rate sele

OLS (36 vars)               0.0398   0.1175   0.0688
Ridge (36 vars)             0.0389   0.1176   0.0688
LASSO (36 vars)             0.0352   0.1178   0.0685
RF (36 vars)                0.0384   0.1176   0.0681
XGBoost (36 vars)           0.0743   0.1154   0.0680


## Setup: Reinstall SHAP

Reinstalls SHAP alongside an upgraded NumPy to resolve a dependency conflict that arose after the earlier NumPy pin.

In [46]:

%pip install shap

  Using cached numpy-1.21.6-cp39-cp39-win_amd64.whl (14.0 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.24.3
    Uninstalling numpy-1.24.3:
      Successfully uninstalled numpy-1.24.3
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
daal4py 2021.5.0 requires daal==2021.4.0, which is not installed.


In [48]:
%pip install numba==0.57.1 llvmlite==0.40.1 --force-reinstall

ERROR: Cannot uninstall 'llvmlite'. It is a distutils installed project and thus we cannot accurately determine which files belong to it which would lead to only a partial uninstall.



  Attempting uninstall: numpy
    Found existing installation: numpy 1.21.6
    Uninstalling numpy-1.21.6:
      Successfully uninstalled numpy-1.21.6
  Attempting uninstall: llvmlite
    Found existing installation: llvmlite 0.38.0


In [49]:
%pip install numpy==1.24.3 --force-reinstall

  Using cached numpy-1.24.3-cp39-cp39-win_amd64.whl (14.9 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.24.4
    Uninstalling numpy-1.24.4:
      Successfully uninstalled numpy-1.24.4



ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
daal4py 2021.5.0 requires daal==2021.4.0, which is not installed.
scipy 1.7.3 requires numpy<1.23.0,>=1.16.5, but you have numpy 1.24.3 which is incompatible.
numba 0.55.1 requires numpy<1.22,>=1.18, but you have numpy 1.24.3 which is incompatible.


In [52]:
%pip install shap==0.44.0

  Using cached numpy-1.21.6-cp39-cp39-win_amd64.whl (14.0 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.24.3
    Uninstalling numpy-1.24.3:
      Successfully uninstalled numpy-1.24.3
  Attempting uninstall: slicer
    Found existing installation: slicer 0.0.8
    Uninstalling slicer-0.0.8:
      Successfully uninstalled slicer-0.0.8
  Attempting uninstall: shap
    Found existing installation: shap 0.49.1
    Uninstalling shap-0.49.1:
      Successfully uninstalled shap-0.49.1
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
daal4py 2021.5.0 requires daal==2021.4.0, which is not installed.


## 11. Model Interpretability — SHAP Analysis

SHAP (SHapley Additive exPlanations) values decompose the XGBoost model's predictions into individual feature contributions. This cell fits XGBoost on the full training history up to the last test month, computes SHAP values for the test set, and saves three plots to disk:

1. **Bar chart** — top 15 predictors ranked by mean absolute SHAP value (global importance)
2. **Beeswarm plot** — distribution of SHAP values across all test observations, showing the direction and magnitude of each feature's effect
3. **Dependence plots** — SHAP value vs feature value scatter plots for the four most important predictors, revealing non-linear relationships

In [3]:
%pip install --upgrade shap

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\vishn\\anaconda3\\Lib\\site-packages\\~hap\\_cext.cp39-win_amd64.pyd'
Consider using the `--user` option or check the permissions.



  Using cached shap-0.49.1-cp39-cp39-win_amd64.whl (547 kB)
  Using cached slicer-0.0.8-py3-none-any.whl (15 kB)
  Attempting uninstall: slicer
    Found existing installation: slicer 0.0.7
    Uninstalling slicer-0.0.7:
      Successfully uninstalled slicer-0.0.7
  Attempting uninstall: shap
    Found existing installation: shap 0.44.0
    Uninstalling shap-0.44.0:
      Successfully uninstalled shap-0.44.0


In [54]:
import dill
import os

save_path = r'C:\Users\vishn\OneDrive\Desktop\RSM MScBA\Thesis\Dill Save\full_session.pkl'

dill.dump_session(save_path)
print("Session saved to:", save_path)
print("File size:", round(os.path.getsize(save_path) / (1024**3), 2), "GB")

Session saved to: C:\Users\vishn\OneDrive\Desktop\RSM MScBA\Thesis\Dill Save\full_session.pkl
File size: 4.53 GB


In [1]:
import dill

save_path = r'C:\Users\vishn\OneDrive\Desktop\RSM MScBA\Thesis\Dill Save\full_session.pkl'

dill.load_session(save_path)
print("Session restored!")

# Verify key variables are back
print("df_model_macro shape:", df_model_macro.shape)
print("Predictors:", len(PREDICTORS_FULL))
print("Test months:", len(test_months))
print("XGBoost results:", len(results_xgb_macro))

Session restored!
df_model_macro shape: (1082801, 42)
Predictors: 36
Test months: 160
XGBoost results: 160


In [5]:
print("SHAP version:", shap.__version__)
print("xgboost version:", xgb.__version__)

SHAP version: 0.44.0
xgboost version: 2.1.4


In [15]:
import shap
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import xgboost as xgb
plt.style.use('default')
matplotlib.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'savefig.facecolor': 'white',
    'text.color':       'black',
    'axes.labelcolor':  'black',
    'xtick.color':      'black',
    'ytick.color':      'black',
    'axes.edgecolor':   'black',
})

# ── Fit XGBoost on final test month ──────────────────────────────────────────

last_test_month = test_months[-1]

train_data = df_model_macro[df_model_macro['date'] < last_test_month]
test_data  = df_model_macro[df_model_macro['date'] == last_test_month]

if len(train_data) > 50000:
    train_data = train_data.sample(n=50000, random_state=42)

X_train = train_data[PREDICTORS_FULL].values
y_train = train_data[TARGET_NEW].values
X_test  = test_data[PREDICTORS_FULL].values
y_test  = test_data[TARGET_NEW].values

print(f"Training observations: {len(X_train):,}")
print(f"Test observations:     {len(X_test):,}")
print(f"Number of predictors:  {len(PREDICTORS_FULL)}")

xgb_shap = xgb.XGBRegressor(
    max_depth=3,
    learning_rate=0.05,
    n_estimators=200,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0
)
xgb_shap.fit(X_train, y_train)
print("\nModel fitted. Computing SHAP values...")

# ── Compute SHAP values ───────────────────────────────────────────────────────

dtest       = xgb.DMatrix(X_test, feature_names=PREDICTORS_FULL)
contribs    = xgb_shap.get_booster().predict(dtest, pred_contribs=True)
shap_values = contribs[:, :-1]
base_value  = contribs[0, -1]

print("SHAP values computed!")
print("SHAP values shape:", shap_values.shape)

# ── Build importance dataframe ────────────────────────────────────────────────

colour_map = {
    'Bond':  'steelblue',
    'Firm':  'coral',
    'Macro': 'mediumseagreen'
}

shap_importance = pd.DataFrame({
    'predictor':     PREDICTORS_FULL,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0),
    'category':      (['Bond']  * len(BOND_PREDICTORS) +
                      ['Firm']  * len(FIRM_PREDICTORS) +
                      ['Macro'] * len(MACRO_PREDICTORS))
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

print("\nSHAP Feature Importance (mean |SHAP value|):")
print(shap_importance.to_string(index=False))

print("\nMacro predictors:")
print(shap_importance[
    shap_importance['category'] == 'Macro'
].to_string(index=False))

print("\nCategory summary:")
print(shap_importance.groupby('category')['mean_abs_shap'].agg(
    ['mean', 'sum', 'count']
).round(5))

# ═════════════════════════════════════════════════════════════════════════════
# PLOT 1 — Bar chart: top 15 predictors colour coded by category
# ═════════════════════════════════════════════════════════════════════════════

top15    = shap_importance.head(15)
colours  = [colour_map[c] for c in top15['category'].values[::-1]]

plt.figure(figsize=(10, 8))
plt.barh(
    top15['predictor'].values[::-1],
    top15['mean_abs_shap'].values[::-1],
    color=colours
)
plt.xlabel('Mean |SHAP Value|')
plt.title('XGBoost SHAP Feature Importance — Top 15 Predictors\n'
          'Bond, Firm and Macro-Financial Variables')
legend_elements = [
    Patch(facecolor='steelblue',      label='Bond characteristics'),
    Patch(facecolor='coral',           label='Firm characteristics'),
    Patch(facecolor='mediumseagreen',  label='Macro-financial variables'),
]
plt.legend(handles=legend_elements, loc='lower right')
plt.tight_layout()
plt.savefig('shap_importance_macro.png', dpi=150, bbox_inches='tight')
plt.close()
print("\nSaved: shap_importance_macro.png")

# ═════════════════════════════════════════════════════════════════════════════
# PLOT 2 — Beeswarm: top 15 predictors
# ═════════════════════════════════════════════════════════════════════════════

idx_sample = np.random.choice(len(X_test), size=2000, replace=False)

plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values[idx_sample],
    X_test[idx_sample],
    feature_names=PREDICTORS_FULL,
    show=False,
    max_display=15
)
plt.tight_layout()
plt.savefig('shap_beeswarm_macro.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: shap_beeswarm_macro.png")

# ═════════════════════════════════════════════════════════════════════════════
# PLOT 3 — Dependence plots: Bond-level predictors (top 6)
# ═════════════════════════════════════════════════════════════════════════════

bond_preds = shap_importance[
    shap_importance['category'] == 'Bond'
].head(6)['predictor'].tolist()

print(f"\nBond predictors for dependence plots: {bond_preds}")

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx_p, predictor in enumerate(bond_preds):
    feat_idx  = PREDICTORS_FULL.index(predictor)
    feat_vals = X_test[:, feat_idx]
    shap_vals = shap_values[:, feat_idx]
    sample_idx = np.random.choice(len(feat_vals), size=3000, replace=False)

    axes[idx_p].scatter(
        feat_vals[sample_idx],
        shap_vals[sample_idx],
        alpha=0.3, s=5, color='steelblue'
    )
    axes[idx_p].axhline(y=0, color='red', linestyle='--', linewidth=0.8)
    axes[idx_p].set_xlabel(predictor)
    axes[idx_p].set_ylabel('SHAP value')
    axes[idx_p].set_title(f'{predictor}')

plt.suptitle('SHAP Dependence Plots — Bond-Level Predictors (Top 6)',
             fontsize=14)
plt.tight_layout()
plt.savefig('shap_dependence_bond.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: shap_dependence_bond.png")

# ═════════════════════════════════════════════════════════════════════════════
# PLOT 4 — Dependence plots: Firm-level predictors (top 6)
# ═════════════════════════════════════════════════════════════════════════════

firm_preds = shap_importance[
    shap_importance['category'] == 'Firm'
].head(6)['predictor'].tolist()

print(f"\nFirm predictors for dependence plots: {firm_preds}")

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx_p, predictor in enumerate(firm_preds):
    feat_idx  = PREDICTORS_FULL.index(predictor)
    feat_vals = X_test[:, feat_idx]
    shap_vals = shap_values[:, feat_idx]
    sample_idx = np.random.choice(len(feat_vals), size=3000, replace=False)

    axes[idx_p].scatter(
        feat_vals[sample_idx],
        shap_vals[sample_idx],
        alpha=0.3, s=5, color='coral'
    )
    axes[idx_p].axhline(y=0, color='red', linestyle='--', linewidth=0.8)
    axes[idx_p].set_xlabel(predictor)
    axes[idx_p].set_ylabel('SHAP value')
    axes[idx_p].set_title(f'{predictor}')

plt.suptitle('SHAP Dependence Plots — Firm-Level Predictors (Top 6)',
             fontsize=14)
plt.tight_layout()
plt.savefig('shap_dependence_firm.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: shap_dependence_firm.png")

# ═════════════════════════════════════════════════════════════════════════════
# PLOT 5 — Dependence plots: Macro-financial predictors (all 8)
# ═════════════════════════════════════════════════════════════════════════════

macro_preds = shap_importance[
    shap_importance['category'] == 'Macro'
].sort_values('mean_abs_shap', ascending=False)['predictor'].tolist()

print(f"\nMacro predictors for dependence plots: {macro_preds}")

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for idx_p, predictor in enumerate(macro_preds):
    feat_idx  = PREDICTORS_FULL.index(predictor)
    feat_vals = X_test[:, feat_idx]
    shap_vals = shap_values[:, feat_idx]
    sample_idx = np.random.choice(len(feat_vals), size=3000, replace=False)

    axes[idx_p].scatter(
        feat_vals[sample_idx],
        shap_vals[sample_idx],
        alpha=0.3, s=5, color='mediumseagreen'
    )
    axes[idx_p].axhline(y=0, color='red', linestyle='--', linewidth=0.8)
    axes[idx_p].set_xlabel(predictor)
    axes[idx_p].set_ylabel('SHAP value')
    axes[idx_p].set_title(f'{predictor}')

plt.suptitle('SHAP Dependence Plots — Macro-Financial Predictors (All 8)',
             fontsize=14)
plt.tight_layout()
plt.savefig('shap_dependence_macro_only.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: shap_dependence_macro_only.png")

# ═════════════════════════════════════════════════════════════════════════════
# Summary
# ═════════════════════════════════════════════════════════════════════════════

print("\nAll SHAP plots saved!")
print("\nFiles created:")
print("  shap_importance_macro.png      — bar chart, top 15, colour coded")
print("  shap_beeswarm_macro.png        — beeswarm, top 15 predictors")
print("  shap_dependence_bond.png       — dependence plots, top 6 bond vars")
print("  shap_dependence_firm.png       — dependence plots, top 6 firm vars")
print("  shap_dependence_macro_only.png — dependence plots, all 8 macro vars")

Training observations: 50,000
Test observations:     5,660
Number of predictors:  36

Model fitted. Computing SHAP values...
SHAP values computed!
SHAP values shape: (5660, 36)

SHAP Feature Importance (mean |SHAP value|):
      predictor  mean_abs_shap category
    swap_spread       0.014724     Bond
 25_mom6mspread       0.010198     Bond
      22_rating       0.004462     Bond
        ret_1_0       0.004160     Firm
          1_age       0.003890     Bond
            dts       0.003854     Bond
      convexity       0.003295     Bond
     6_duration       0.002859     Bond
       ret_12_1       0.002261     Firm
     def_spread       0.001537    Macro
zero_trades_21d       0.001449     Firm
          at_me       0.001222     Firm
       2_coupon       0.001205     Bond
  market_equity       0.001200     Firm
          tsy10       0.000987    Macro
            vix       0.000897    Macro
        debt_me       0.000887     Firm
         vix_ch       0.000796    Macro
  27_volatility  

## 12. Subgroup Analysis — Investment Grade vs High Yield

This cell disaggregates XGBoost's out-of-sample R² by credit segment. IG bonds (`c0a0`) and HY bonds (`H0A0`) are evaluated separately to test whether predictability differs by credit quality — a common finding in the empirical credit risk literature, where HY spreads tend to be more volatile and potentially harder to predict.

In [8]:
import numpy as np
import pandas as pd

# ── IG vs HY subgroup analysis ────────────────────────────────────────────────

print("=" * 55)
print("SUBGROUP ANALYSIS — XGBoost (36 predictors)")
print("=" * 55)

print("\n--- IG vs HY ---")

for group_name, group_filter in [('IG', 'c0a0'), ('HY', 'H0A0')]:

    group_results = []

    for r in results_xgb_macro:          # changed: results_xgb → results_xgb_macro
        month     = r['month']
        full_test = df_model_macro[       # changed: df_model → df_model_macro
            df_model_macro['date'] == month
        ]

        if len(full_test) == 0:
            continue

        group_mask = full_test['ice_idx'].values == group_filter

        if group_mask.sum() < 10:
            continue

        y_group    = r['y_test'][group_mask]
        pred_group = r['pred'][group_mask]

        group_results.append({
            'y_test': y_group,
            'pred':   pred_group
        })

    all_y    = np.concatenate([r['y_test'] for r in group_results])
    all_pred = np.concatenate([r['pred']   for r in group_results])
    ss_res   = np.sum((all_y - all_pred) ** 2)
    ss_tot   = np.sum(all_y ** 2)
    r2       = 1 - ss_res / ss_tot

    print(f"{group_name}: OOS R² = {r2:.4f}, N = {len(all_y):,}")

# ── Regime splits ─────────────────────────────────────────────────────────────

print("\n--- Temporal regime splits ---")

COVID_START = pd.Timestamp('2020-03-31')
MIDPOINT    = pd.Timestamp('2015-06-30')

for regime_name, date_filter in [
    ('Pre-COVID  (2009-2020)', lambda m: m < COVID_START),
    ('Post-COVID (2020-2022)', lambda m: m >= COVID_START),
]:
    regime_results = [
        r for r in results_xgb_macro    # changed: results_xgb → results_xgb_macro
        if date_filter(r['month'])
    ]

    if len(regime_results) == 0:
        print(f"{regime_name}: No results found")
        continue

    all_y    = np.concatenate([r['y_test'] for r in regime_results])
    all_pred = np.concatenate([r['pred']   for r in regime_results])
    ss_res   = np.sum((all_y - all_pred) ** 2)
    ss_tot   = np.sum(all_y ** 2)
    r2       = 1 - ss_res / ss_tot

    print(f"{regime_name}: OOS R² = {r2:.4f}, "
          f"months = {len(regime_results)}, "
          f"N = {len(all_y):,}")

for regime_name, date_filter in [
    ('Early post-GFC (2009-2015)', lambda m: m < MIDPOINT),
    ('Late post-GFC  (2015-2022)', lambda m: m >= MIDPOINT),
]:
    regime_results = [
        r for r in results_xgb_macro    # changed: results_xgb → results_xgb_macro
        if date_filter(r['month'])
    ]

    if len(regime_results) == 0:
        print(f"{regime_name}: No results found")
        continue

    all_y    = np.concatenate([r['y_test'] for r in regime_results])
    all_pred = np.concatenate([r['pred']   for r in regime_results])
    ss_res   = np.sum((all_y - all_pred) ** 2)
    ss_tot   = np.sum(all_y ** 2)
    r2       = 1 - ss_res / ss_tot

    print(f"{regime_name}: OOS R² = {r2:.4f}, "
          f"months = {len(regime_results)}, "
          f"N = {len(all_y):,}")

# ── Summary table ─────────────────────────────────────────────────────────────

print("\n" + "=" * 55)
print("SUBGROUP SUMMARY TABLE")
print("=" * 55)
print(f"{'Subgroup':<30} {'OOS R²':>8} {'N':>10}")
print("-" * 55)
print(f"{'Full sample':<30} {'0.0743':>8} {'796,546':>10}")
print("(run above code and fill in IG, HY, regime numbers)")

SUBGROUP ANALYSIS — XGBoost (36 predictors)

--- IG vs HY ---
IG: OOS R² = 0.0468, N = 657,081
HY: OOS R² = 0.1310, N = 139,465

--- Temporal regime splits ---
Pre-COVID  (2009-2020): OOS R² = 0.0748, months = 128, N = 611,331
Post-COVID (2020-2022): OOS R² = 0.0734, months = 32, N = 185,215
Early post-GFC (2009-2015): OOS R² = 0.0566, months = 71, N = 301,438
Late post-GFC  (2015-2022): OOS R² = 0.0805, months = 89, N = 495,108

SUBGROUP SUMMARY TABLE
Subgroup                         OOS R²          N
-------------------------------------------------------
Full sample                      0.0743    796,546
(run above code and fill in IG, HY, regime numbers)


## 13. Regime Analysis — Pre vs Post COVID, Early vs Late Post-GFC

This cell examines whether model performance is stable across economic regimes. Two splits are tested: (1) pre- vs post-COVID (March 2020 as the breakpoint), given the unprecedented spread widening during the pandemic; and (2) an early vs late post-GFC split around mid-2015, to check for any trend in predictability over the sample period.

In [9]:
import numpy as np
import pandas as pd

# ── Pick three interesting months to examine ──────────────────────────────────

crisis_months = {
    'COVID crash (Mar 2020)':    pd.Timestamp('2020-03-31'),
    'Post-COVID recovery (Jun 2020)': pd.Timestamp('2020-06-30'),
    'Rate hike shock (Jun 2022)': pd.Timestamp('2022-06-30'),
}

print("=" * 70)
print("CRISIS MONTH PREDICTION ANALYSIS")
print("=" * 70)

for crisis_name, crisis_month in crisis_months.items():

    # Find closest available test month
    available = [m for m in test_months if m >= crisis_month]
    if not available:
        print(f"\n{crisis_name}: month not in test period")
        continue
    actual_month = min(available)

    # Get OLS predictions for this month
    ols_result = next(
        (r for r in results_ols_macro if r['month'] == actual_month),
        None
    )
    # Get XGBoost predictions for this month
    xgb_result = next(
        (r for r in results_xgb_macro if r['month'] == actual_month),
        None
    )

    if ols_result is None or xgb_result is None:
        print(f"\n{crisis_name}: results not found")
        continue

    y_true    = xgb_result['y_test']
    ols_pred  = ols_result['pred']
    xgb_pred  = xgb_result['pred']

    # Compute monthly R²
    def monthly_r2(y, yhat):
        ss_res = np.sum((y - yhat) ** 2)
        ss_tot = np.sum(y ** 2)
        return 1 - ss_res / ss_tot

    ols_r2 = monthly_r2(y_true, ols_pred)
    xgb_r2 = monthly_r2(y_true, xgb_pred)

    # Directional accuracy — did the model predict widening/tightening correctly?
    ols_dir  = np.mean(np.sign(ols_pred) == np.sign(y_true))
    xgb_dir  = np.mean(np.sign(xgb_pred) == np.sign(y_true))

    print(f"\n{crisis_name} ({actual_month.strftime('%Y-%m')})")
    print(f"  Bonds in cross-section: {len(y_true):,}")
    print(f"  Actual spread change — mean: {y_true.mean():.4f}, "
          f"std: {y_true.std():.4f}")
    print(f"  OLS   — R²: {ols_r2:.4f}, "
          f"Directional accuracy: {ols_dir:.1%}")
    print(f"  XGBoost — R²: {xgb_r2:.4f}, "
          f"Directional accuracy: {xgb_dir:.1%}")
    print(f"  XGBoost advantage: {xgb_r2 - ols_r2:+.4f} R² points")

CRISIS MONTH PREDICTION ANALYSIS

COVID crash (Mar 2020) (2020-03)
  Bonds in cross-section: 5,321
  Actual spread change — mean: 0.0139, std: 0.2934
  OLS   — R²: 0.1240, Directional accuracy: 72.4%
  XGBoost — R²: 0.0803, Directional accuracy: 54.3%
  XGBoost advantage: -0.0437 R² points

Post-COVID recovery (Jun 2020) (2020-06)
  Bonds in cross-section: 5,762
  Actual spread change — mean: -0.0041, std: 0.1343
  OLS   — R²: 0.0735, Directional accuracy: 54.9%
  XGBoost — R²: 0.0719, Directional accuracy: 56.6%
  XGBoost advantage: -0.0016 R² points

Rate hike shock (Jun 2022) (2022-06)
  Bonds in cross-section: 5,830
  Actual spread change — mean: -0.0049, std: 0.1263
  OLS   — R²: 0.0757, Directional accuracy: 53.6%
  XGBoost — R²: 0.0834, Directional accuracy: 57.2%
  XGBoost advantage: +0.0077 R² points


In [10]:
import pandas as pd
import numpy as np

# ── Generate predictions for the last test month ──────────────────────────────

last_month = test_months[-1]

# Get the last month's test data with bond identifiers
last_test_data = df_model_macro[
    df_model_macro['date'] == last_month
].copy()

# Get XGBoost predictions for last month
xgb_last = next(
    r for r in results_xgb_macro if r['month'] == last_month
)
ols_last = next(
    r for r in results_ols_macro if r['month'] == last_month
)

# Add predictions to the dataframe
last_test_data = last_test_data.reset_index(drop=True)
last_test_data['xgb_pred']      = xgb_last['pred']
last_test_data['ols_pred']       = ols_last['pred']
last_test_data['actual_change']  = xgb_last['y_test']
last_test_data['xgb_error']      = (
    last_test_data['actual_change'] - last_test_data['xgb_pred']
)
last_test_data['ols_error']      = (
    last_test_data['actual_change'] - last_test_data['ols_pred']
)

print(f"Predictions for: {last_month.strftime('%B %Y')}")
print(f"Bonds in cross-section: {len(last_test_data):,}")

# ── Top 20 predicted to widen most (buy protection / underweight) ─────────────

print("\n" + "=" * 65)
print("TOP 20 BONDS PREDICTED TO WIDEN (XGBoost)")
print("High spread change = spread widening relative to peers")
print("=" * 65)

top_widen = last_test_data.nlargest(20, 'xgb_pred')[
    ['cusip', 'RATING_NUM', 'ice_idx', '18_spread',
     'xgb_pred', 'actual_change']
]
top_widen.columns = ['CUSIP', 'Rating', 'IG/HY', 'Current Spread Rank',
                     'XGB Predicted Change', 'Actual Change']
print(top_widen.to_string(index=False))

# ── Top 20 predicted to tighten most (overweight) ────────────────────────────

print("\n" + "=" * 65)
print("TOP 20 BONDS PREDICTED TO TIGHTEN (XGBoost)")
print("Negative spread change = spread tightening relative to peers")
print("=" * 65)

top_tighten = last_test_data.nsmallest(20, 'xgb_pred')[
    ['cusip', 'RATING_NUM', 'ice_idx', '18_spread',
     'xgb_pred', 'actual_change']
]
top_tighten.columns = ['CUSIP', 'Rating', 'IG/HY', 'Current Spread Rank',
                       'XGB Predicted Change', 'Actual Change']
print(top_tighten.to_string(index=False))

# ── Prediction accuracy summary ───────────────────────────────────────────────

print("\n" + "=" * 65)
print("PREDICTION ACCURACY SUMMARY")
print("=" * 65)

# Directional accuracy
xgb_dir = np.mean(
    np.sign(last_test_data['xgb_pred']) ==
    np.sign(last_test_data['actual_change'])
)
ols_dir = np.mean(
    np.sign(last_test_data['ols_pred']) ==
    np.sign(last_test_data['actual_change'])
)

# Top/bottom decile accuracy
n = len(last_test_data)
decile = n // 10

# Sort by predicted change
xgb_sorted = last_test_data.sort_values('xgb_pred')
ols_sorted  = last_test_data.sort_values('ols_pred')

# Check if top decile (predicted wideners) actually widened
xgb_top_acc = np.mean(
    xgb_sorted.tail(decile)['actual_change'] > 0
)
ols_top_acc = np.mean(
    ols_sorted.tail(decile)['actual_change'] > 0
)

# Check if bottom decile (predicted tighteners) actually tightened
xgb_bot_acc = np.mean(
    xgb_sorted.head(decile)['actual_change'] < 0
)
ols_bot_acc = np.mean(
    ols_sorted.head(decile)['actual_change'] < 0
)

print(f"{'Metric':<40} {'OLS':>8} {'XGBoost':>8}")
print("-" * 60)
print(f"{'Directional accuracy (full sample)':<40} "
      f"{ols_dir:>8.1%} {xgb_dir:>8.1%}")
print(f"{'Top decile accuracy (predicted wideners)':<40} "
      f"{ols_top_acc:>8.1%} {xgb_top_acc:>8.1%}")
print(f"{'Bottom decile accuracy (predicted tighteners)':<40} "
      f"{ols_bot_acc:>8.1%} {xgb_bot_acc:>8.1%}")

# ── IG vs HY prediction breakdown ────────────────────────────────────────────

print("\n" + "=" * 65)
print("PREDICTION ACCURACY BY SEGMENT")
print("=" * 65)

for segment, filter_val in [('Investment Grade', 'c0a0'),
                              ('High Yield', 'H0A0')]:
    seg = last_test_data[last_test_data['ice_idx'] == filter_val]
    if len(seg) < 10:
        continue

    xgb_seg_dir = np.mean(
        np.sign(seg['xgb_pred']) == np.sign(seg['actual_change'])
    )
    ols_seg_dir = np.mean(
        np.sign(seg['ols_pred']) == np.sign(seg['actual_change'])
    )

    print(f"\n{segment} (n={len(seg):,}):")
    print(f"  OLS directional accuracy:     {ols_seg_dir:.1%}")
    print(f"  XGBoost directional accuracy: {xgb_seg_dir:.1%}")
    print(f"  XGBoost advantage:            "
          f"{xgb_seg_dir - ols_seg_dir:+.1%}")

Predictions for: October 2022
Bonds in cross-section: 5,660

TOP 20 BONDS PREDICTED TO WIDEN (XGBoost)
High spread change = spread widening relative to peers
    CUSIP  Rating IG/HY  Current Spread Rank  XGB Predicted Change  Actual Change
023135BN5       4  c0a0            -0.991127              0.349300       0.006886
05348EAT6       7  c0a0            -0.943981              0.333741       0.193149
023135AN6       4  c0a0            -0.987996              0.267407       0.014436
382550BH3      14  H0A0             0.081072              0.260091      -0.253896
89236TFS9       5  c0a0            -0.972860              0.258259       0.013834
487836BS6       9  c0a0            -0.998260              0.257348       0.125909
023135CD6       4  c0a0            -0.992519              0.247746       0.007052
023135BW5       4  c0a0            -0.985038              0.213606      -0.002004
092113AH2       8  c0a0            -0.914405              0.213565       0.202446
023135CE4       4  c0a

In [11]:
import pandas as pd
import numpy as np

# ── Step 1: Find bonds from well-known companies ──────────────────────────────

# Search for recognisable issuers in your dataset
# The issuer_cusip is the first 6 digits of cusip

# Load the original df to get issuer names if available
print("Available columns for identifying issuers:")
print([c for c in df.columns if any(x in c.lower() 
      for x in ['name', 'issuer', 'ticker', 'company'])])

# Check what identifier columns we have
print("\nSample of identifier columns:")
print(df[['cusip', 'issuer_cusip', 'gvkey', 'permno']].head(10))

# ── Step 2: Find bonds with longest history in test period ────────────────────

# Count how many test months each bond appears in
bond_coverage = df_model_macro[
    df_model_macro['date'] >= pd.Timestamp('2009-07-31')
].groupby('cusip').agg(
    months = ('date', 'count'),
    avg_spread = ('18_spread', 'mean'),
    rating = ('RATING_NUM', 'mean'),
    ice_idx = ('ice_idx', 'first')
).reset_index()

bond_coverage = bond_coverage.sort_values('months', ascending=False)

print("\nBonds with most complete test period coverage:")
print(bond_coverage.head(20).to_string(index=False))

print("\nIG bonds with most coverage:")
print(bond_coverage[bond_coverage['ice_idx'] == 'c0a0'].head(10).to_string(index=False))

print("\nHY bonds with most coverage:")
print(bond_coverage[bond_coverage['ice_idx'] == 'H0A0'].head(10).to_string(index=False))

Available columns for identifying issuers:
['issuer_cusip']

Sample of identifier columns:
       cusip issuer_cusip    gvkey   permno
0  001031AC7       001031  11903.0  10025.0
1  001031AC7       001031  11903.0  10025.0
2  001031AC7       001031  11903.0  10025.0
3  001031AC7       001031  11903.0  10025.0
4  001031AC7       001031  11903.0  10025.0
5  001031AC7       001031  11903.0  10025.0
6  001031AC7       001031  11903.0  10025.0
7  001031AC7       001031  11903.0  10025.0
8  001031AC7       001031  11903.0  10025.0
9  001031AC7       001031  11903.0  10025.0

Bonds with most complete test period coverage:
    cusip  months  avg_spread   rating ice_idx
17275RAD4     160   -0.107678  5.00000    c0a0
156686AM9     160    0.831293 12.19375    c0a0
013817AK7     160    0.711470 10.64375    c0a0
013817AJ0     160    0.653514 10.64375    c0a0
20030NAV3     160    0.260907  7.26875    c0a0
20030NAY7     160    0.234589  7.26875    c0a0
969457BB5     160    0.658429 10.08750    c0a0
9

In [13]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ── Step 1: Identify what these CUSIPs are ────────────────────────────────────
# CUSIP first 6 digits = issuer, last 2 = issue number
# Let's look at the issuer_cusip to identify companies

# Pick one bond from each category to investigate
selected_cusips = {
    'IG_AAA':  '17275RAD4',  # rating ~5 = A range
    'IG_BBB1': '156686AM9',  # rating ~12 = BBB range  
    'IG_BBB2': '013817AK7',  # rating ~10 = BBB range
    'HY_BB1':  '532716AK3',  # rating ~12 = BB range
    'HY_B':    '88033GAV2',  # rating ~16 = B range
}

# Check their history in the dataset
for label, cusip in selected_cusips.items():
    bond_data = df_model_macro[
        df_model_macro['cusip'] == cusip
    ][['date', 'RATING_NUM', 'ice_idx', 
       '18_spread', 'spread_change',
       '6_duration', '22_rating']].head(5)
    print(f"\n{label} — CUSIP: {cusip}")
    print(bond_data.to_string(index=False))

# ── Step 2: Build prediction history for selected bonds ───────────────────────

def get_bond_predictions(cusip, results_xgb, results_ols, df_model_macro):
    """Extract month-by-month predictions for a specific bond."""
    records = []
    
    for xgb_r, ols_r in zip(results_xgb_macro, results_ols_macro):
        month     = xgb_r['month']
        full_test = df_model_macro[df_model_macro['date'] == month]
        full_test = full_test.reset_index(drop=True)
        
        # Find this bond in the cross-section
        bond_idx = full_test[full_test['cusip'] == cusip].index
        
        if len(bond_idx) == 0:
            continue
            
        idx = bond_idx[0]
        
        records.append({
            'date':          month,
            'actual_change': xgb_r['y_test'][idx],
            'xgb_pred':      xgb_r['pred'][idx],
            'ols_pred':       ols_r['pred'][idx],
            'spread_level':  full_test.loc[idx, '18_spread'],
            'rating':        full_test.loc[idx, 'RATING_NUM'],
        })
    
    return pd.DataFrame(records)

# ── Step 3: Get predictions for each selected bond ────────────────────────────

print("\nBuilding prediction histories...")

bond_histories = {}
for label, cusip in selected_cusips.items():
    hist = get_bond_predictions(
        cusip, results_xgb_macro, results_ols_macro, df_model_macro
    )
    bond_histories[label] = hist
    print(f"{label} ({cusip}): {len(hist)} months of predictions")

# ── Step 4: Summary statistics per bond ──────────────────────────────────────

print("\n" + "=" * 75)
print("PREDICTION ACCURACY BY BOND")
print("=" * 75)
print(f"{'Bond':<12} {'CUSIP':<12} {'Months':>7} "
      f"{'OLS R²':>8} {'XGB R²':>8} "
      f"{'OLS Dir%':>9} {'XGB Dir%':>9}")
print("-" * 75)

for label, cusip in selected_cusips.items():
    hist = bond_histories[label]
    if len(hist) < 10:
        continue
    
    y      = hist['actual_change'].values
    ols_p  = hist['ols_pred'].values
    xgb_p  = hist['xgb_pred'].values
    
    # R²
    ols_r2 = 1 - np.sum((y - ols_p)**2) / np.sum(y**2)
    xgb_r2 = 1 - np.sum((y - xgb_p)**2) / np.sum(y**2)
    
    # Directional accuracy
    ols_dir = np.mean(np.sign(ols_p) == np.sign(y))
    xgb_dir = np.mean(np.sign(xgb_p) == np.sign(y))
    
    print(f"{label:<12} {cusip:<12} {len(hist):>7} "
          f"{ols_r2:>8.4f} {xgb_r2:>8.4f} "
          f"{ols_dir:>9.1%} {xgb_dir:>9.1%}")

# ── Step 5: Plot predicted vs actual for best covered bond ────────────────────



# ── Plot 1: Full history for main bond ───────────────────────────────────────

main_label = 'IG_BBB1'
main_cusip = selected_cusips[main_label]
hist = bond_histories[main_label]

# Convert to numpy to fix matplotlib version issue
dates      = hist['date'].values
actual     = hist['actual_change'].values
xgb_pred   = hist['xgb_pred'].values
ols_pred   = hist['ols_pred'].values
spread_lvl = hist['spread_level'].values
xgb_err    = np.abs(actual - xgb_pred)
ols_err    = np.abs(actual - ols_pred)

fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Panel 1 — predicted vs actual
axes[0].plot(dates, actual,
             label='Actual', color='black', linewidth=1.5)
axes[0].plot(dates, xgb_pred,
             label='XGBoost', color='steelblue',
             linewidth=1, linestyle='--', alpha=0.8)
axes[0].plot(dates, ols_pred,
             label='OLS', color='coral',
             linewidth=1, linestyle=':', alpha=0.8)
axes[0].axhline(y=0, color='grey', linewidth=0.5)
axes[0].set_title(
    f'Predicted vs Actual Spread Changes — {main_label} ({main_cusip})'
)
axes[0].set_ylabel('Spread Change')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Panel 2 — spread level
axes[1].plot(dates, spread_lvl, color='darkgreen', linewidth=1.5)
axes[1].set_title('Spread Level (cross-sectional rank)')
axes[1].set_ylabel('18_spread')
axes[1].grid(True, alpha=0.3)

# Panel 3 — prediction errors
axes[2].plot(dates, xgb_err,
             label='|XGBoost error|', color='steelblue',
             linewidth=1, alpha=0.8)
axes[2].plot(dates, ols_err,
             label='|OLS error|', color='coral',
             linewidth=1, alpha=0.8)
axes[2].set_title('Absolute Prediction Error — XGBoost vs OLS')
axes[2].set_ylabel('|Actual − Predicted|')
axes[2].set_xlabel('Date')
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
save_path = (r'C:\Users\vishn\OneDrive\Desktop\RSM MScBA'
             r'\Thesis\ThesisCode\bond_predictions.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"Plot saved: {save_path}")

# ── Plot 2: Zoom into COVID and rate hike ─────────────────────────────────────

covid_mask = (
    (hist['date'] >= pd.Timestamp('2020-01-31')) &
    (hist['date'] <= pd.Timestamp('2020-12-31'))
)
rate_mask = (
    (hist['date'] >= pd.Timestamp('2022-01-31')) &
    (hist['date'] <= pd.Timestamp('2022-10-31'))
)

covid_hist = hist[covid_mask]
rate_hist  = hist[rate_mask]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, window, title in [
    (axes[0], covid_hist, 'COVID Period (2020)'),
    (axes[1], rate_hist,  'Rate Hike Period (2022)')
]:
    d = window['date'].values
    a = window['actual_change'].values
    x = window['xgb_pred'].values
    o = window['ols_pred'].values

    ax.plot(d, a, label='Actual', color='black', linewidth=2)
    ax.plot(d, x, label='XGBoost', color='steelblue',
            linewidth=1.5, linestyle='--')
    ax.plot(d, o, label='OLS', color='coral',
            linewidth=1.5, linestyle=':')
    ax.axhline(y=0, color='grey', linewidth=0.5)
    ax.set_title(title)
    ax.set_ylabel('Spread Change')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
save_path2 = (r'C:\Users\vishn\OneDrive\Desktop\RSM MScBA'
              r'\Thesis\ThesisCode\bond_predictions_zoom.png')
plt.savefig(save_path2, dpi=150, bbox_inches='tight')
plt.close()
print(f"Zoom plot saved: {save_path2}")


IG_AAA — CUSIP: 17275RAD4
      date  RATING_NUM ice_idx  18_spread  spread_change  6_duration  22_rating
2009-02-28           5    c0a0  -0.726400       0.032297    0.972308  -0.819453
2009-03-31           5    c0a0  -0.694103      -0.062458    0.965923  -0.819363
2009-04-30           5    c0a0  -0.756561       0.008819    0.979267  -0.825406
2009-05-31           5    c0a0  -0.747742       0.111405    0.978723  -0.822222
2009-06-30           5    c0a0  -0.636337      -0.087108    0.953706  -0.821067

IG_BBB1 — CUSIP: 156686AM9
      date  RATING_NUM ice_idx  18_spread  spread_change  6_duration  22_rating
2002-07-31           9    c0a0   0.351950       0.033257    0.831639   0.181629
2002-08-31           9    c0a0   0.385207      -0.032612    0.808669   0.195220
2002-09-30           9    c0a0   0.352595      -0.262125    0.823086   0.179064
2002-10-31           9    c0a0   0.090470      -0.196147    0.874540   0.149955
2002-11-30           9    c0a0  -0.105677       0.004405    0.880